In [8]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\vinja\Projects\llm_engineering\.venv\Scripts\python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

VALIDATION_FILE = "VIIRS_60_Event_Validation_Review_Pack.xlsx"
V11_FILE = "viirs_v11_final_event_dataset.csv"

In [10]:
xl = pd.ExcelFile(VALIDATION_FILE)

print("Sheets:")
print(xl.sheet_names)

validation = pd.read_excel(
    VALIDATION_FILE,
    sheet_name=xl.sheet_names[0]
)

print("\nValidation shape:", validation.shape)
display(validation.head())

Sheets:
['Validation Review', 'Reviewer Guide', 'Checklist']

Validation shape: (60, 31)


,review_id,event_id,latitude,longitude,start_date,end_date,active_days,duration_days,detection_count,daily_object_count,spatial_diameter_km,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4,mean_bright_ti5,max_bright_ti5,temporal_group,satellite_map_link,osm_map_link,actual_class,confidence,industrial_evidence,agricultural_evidence,natural_evidence,other_evidence,evidence_notes,imagery_checked,osm_checked,review_date,reviewer_notes
0,1,12,21.968755,85.314067,2024-01-01,2024-01-17,17,17,26,19,0.539481,0.973846,1.51,301.660385,310.16,285.216154,288.39,Persistent (15+ active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=21.9687546...,Industrial,High,Steel Plant,none,none,Roads/settlement,Industrial complex; persistent activity,"Yes - 500m, 1km",yes,2026-09-03,Exact facility boundary unclear
1,2,1347,22.265312,75.133081,2024-01-10,2024-01-31,20,22,30,27,0.688072,1.218333,1.91,303.530000,312.84,287.886000,291.80,Persistent (15+ active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=22.2653123...,Industrial,high,Cement plant,none,none,none,Cement plant at event location; persistent the...,"Yes - 500m, 1km",yes,2026-09-03,Strong industrial context
2,3,1967,15.842830,75.435250,2024-01-13,2024-01-13,1,1,1,1,0.000000,1.210000,1.21,305.500000,305.50,289.620000,289.62,Transient (1–2 active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=15.84283&m...,Unknown,low,none,Cropland; no clear burn signature,none,none,Transient hotspot in cropland; no clear burn s...,"Yes - 500m, 1km",yes,2026-09-03,Source cannot be confidently attributed
3,4,479,20.091620,79.026615,2024-01-03,2024-01-04,2,2,2,2,0.259333,0.790000,0.96,303.415000,305.92,291.045000,291.20,Transient (1–2 active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=20.09162&m...,Industrial,Medium,Coal mine,none,none,none,Open-pit mine at event location; mapped mining...,"Yes - 500m, 1km",yes,2026-09-03,Exact thermal source unclear
4,5,368,27.558151,96.039266,2024-01-02,2024-01-27,19,26,22,19,0.482276,0.625455,1.09,301.573182,317.51,281.559545,283.52,Persistent (15+ active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=27.5581513...,Unknown,Low,none,none,none,Rural settlement/roads,Mixed rural landscape; no clear source identified,"Yes - 500m, 1km",yes,2026-09-03,Persistent activity but source unclear


In [11]:
print("Validation columns:")
for col in validation.columns:
    print("-", col)

Validation columns:
- review_id
- event_id
- latitude
- longitude
- start_date
- end_date
- active_days
- duration_days
- detection_count
- daily_object_count
- spatial_diameter_km
- mean_frp
- max_frp
- mean_bright_ti4
- max_bright_ti4
- mean_bright_ti5
- max_bright_ti5
- temporal_group
- satellite_map_link
- osm_map_link
- actual_class
- confidence
- industrial_evidence
- agricultural_evidence
- natural_evidence
- other_evidence
- evidence_notes
- imagery_checked
- osm_checked
- review_date
- reviewer_notes


In [12]:
print("Total validation rows:", len(validation))

completed = validation[
    validation["actual_class"].notna() &
    validation["actual_class"].astype(str).str.strip().ne("")
].copy()

print("Completed labels:", len(completed))
print("Remaining unlabeled:", len(validation) - len(completed))

display(
    completed[
        ["event_id", "actual_class", "confidence"]
    ]
)

Total validation rows: 60
Completed labels: 18
Remaining unlabeled: 42


,event_id,actual_class,confidence
0,12,Industrial,High
1,1347,Industrial,high
2,1967,Unknown,low
3,479,Industrial,Medium
4,368,Unknown,Low
5,3859,Unknown,Low
6,2201,Natural_Vegetation,medium
7,2882,Natural_Vegetation,high
8,172,Industrial,high
9,693,Industrial,high


In [13]:
print("Actual class distribution:")
display(
    completed["actual_class"]
    .value_counts(dropna=False)
    .rename_axis("class")
    .reset_index(name="count")
)

print("\nConfidence distribution:")
display(
    completed["confidence"]
    .value_counts(dropna=False)
    .rename_axis("confidence")
    .reset_index(name="count")
)

Actual class distribution:


,class,count
0,Industrial,8
1,Natural_Vegetation,4
2,Unknown,3
3,Industrial,3



Confidence distribution:


,confidence,count
0,high,9
1,medium,3
2,Low,2
3,Medium,2
4,High,1
5,low,1


In [14]:
allowed_classes = {
    "Industrial",
    "Agricultural",
    "Natural_Vegetation",
    "Other",
    "Unknown"
}

allowed_confidence = {
    "High",
    "Medium",
    "Low"
}

print("Duplicate event IDs:",
      completed["event_id"].duplicated().sum())

print("Invalid classes:",
      (~completed["actual_class"].isin(allowed_classes)).sum())

print("Invalid confidence:",
      (~completed["confidence"].isin(allowed_confidence)).sum())

print("Missing event IDs:",
      completed["event_id"].isna().sum())

Duplicate event IDs: 0
Invalid classes: 3
Invalid confidence: 13
Missing event IDs: 0


In [15]:
v11 = pd.read_csv(V11_FILE)

print("V11 shape:", v11.shape)
display(v11.head())

V11 shape: (4893, 24)


,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5,start_date,end_date,active_days,detection_count,duration_days,activity_frequency,detections_per_active_day,centroid_lat,centroid_lon,daily_object_count,spatial_diameter_km,frp_range,ti4_range,ti5_range
0,0,1.635412,4.97,0.946512,307.793529,337.48,10.981464,284.200706,289.96,3.564605,2024-01-01,2024-01-31,27,85,31,0.870968,3.148148,23.169481,82.341132,75,1.545094,3.334588,29.686471,5.759294
1,1,1.433333,2.32,0.780534,302.420000,306.63,4.874249,286.516667,286.93,0.362951,2024-01-01,2024-01-02,2,3,2,1.000000,1.500000,24.209340,82.712480,2,0.367945,0.886667,4.210000,0.413333
2,2,1.735529,3.90,0.727670,309.442824,330.20,8.531087,289.782588,296.08,2.653108,2024-01-01,2024-01-31,27,85,31,0.870968,3.148148,22.053932,88.124138,64,1.021891,2.164471,20.757176,6.297412
3,3,1.436667,2.07,0.563678,302.433333,307.31,4.806603,285.616667,286.50,0.784750,2024-01-01,2024-01-02,2,3,2,1.000000,1.500000,24.204233,82.711530,2,0.349574,0.633333,4.876667,0.883333
4,4,0.800000,0.80,0.000000,302.095000,302.97,1.237437,288.415000,288.52,0.148492,2024-01-01,2024-01-01,1,2,1,1.000000,2.000000,22.319650,82.566480,1,0.357913,0.000000,0.875000,0.105000


In [16]:
completed["event_id"] = pd.to_numeric(
    completed["event_id"],
    errors="coerce"
).astype("Int64")

v11["event_id"] = pd.to_numeric(
    v11["event_id"],
    errors="coerce"
).astype("Int64")

validation_ids = set(completed["event_id"].dropna())
v11_ids = set(v11["event_id"].dropna())

print("Validation events:", len(validation_ids))
print("Matched in V11:", len(validation_ids & v11_ids))
print("Missing from V11:", len(validation_ids - v11_ids))

Validation events: 18
Matched in V11: 18
Missing from V11: 0


In [17]:
validation_ml = completed.merge(
    v11,
    on="event_id",
    how="left",
    suffixes=("_validation", "")
)

print("Shape:", validation_ml.shape)

print(
    "Rows without V11 features:",
    validation_ml["mean_frp"].isna().sum()
)

display(validation_ml.head())

Shape: (18, 54)
Rows without V11 features: 0


,review_id,event_id,latitude,longitude,start_date_validation,end_date_validation,active_days_validation,duration_days_validation,detection_count_validation,daily_object_count_validation,spatial_diameter_km_validation,mean_frp_validation,max_frp_validation,mean_bright_ti4_validation,max_bright_ti4_validation,mean_bright_ti5_validation,max_bright_ti5_validation,temporal_group,satellite_map_link,osm_map_link,actual_class,confidence,industrial_evidence,agricultural_evidence,natural_evidence,other_evidence,evidence_notes,imagery_checked,osm_checked,review_date,reviewer_notes,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5,start_date,end_date,active_days,detection_count,duration_days,activity_frequency,detections_per_active_day,centroid_lat,centroid_lon,daily_object_count,spatial_diameter_km,frp_range,ti4_range,ti5_range
0,1,12,21.968755,85.314067,2024-01-01,2024-01-17,17,17,26,19,0.539481,0.973846,1.51,301.660385,310.16,285.216154,288.39,Persistent (15+ active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=21.9687546...,Industrial,High,Steel Plant,none,none,Roads/settlement,Industrial complex; persistent activity,"Yes - 500m, 1km",yes,2026-09-03,Exact facility boundary unclear,0.973846,1.51,0.352784,301.660385,310.16,4.639727,285.216154,288.39,1.640303,2024-01-01,2024-01-17,17,26,17,1.000000,1.529412,21.968755,85.314067,19,0.539481,0.536154,8.499615,3.173846
1,2,1347,22.265312,75.133081,2024-01-10,2024-01-31,20,22,30,27,0.688072,1.218333,1.91,303.530000,312.84,287.886000,291.80,Persistent (15+ active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=22.2653123...,Industrial,high,Cement plant,none,none,none,Cement plant at event location; persistent the...,"Yes - 500m, 1km",yes,2026-09-03,Strong industrial context,1.218333,1.91,0.411960,303.530000,312.84,4.580089,287.886000,291.80,3.314337,2024-01-10,2024-01-31,20,30,22,0.909091,1.500000,22.265312,75.133081,27,0.688072,0.691667,9.310000,3.914000
2,3,1967,15.842830,75.435250,2024-01-13,2024-01-13,1,1,1,1,0.000000,1.210000,1.21,305.500000,305.50,289.620000,289.62,Transient (1–2 active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=15.84283&m...,Unknown,low,none,Cropland; no clear burn signature,none,none,Transient hotspot in cropland; no clear burn s...,"Yes - 500m, 1km",yes,2026-09-03,Source cannot be confidently attributed,1.210000,1.21,0.000000,305.500000,305.50,0.000000,289.620000,289.62,0.000000,2024-01-13,2024-01-13,1,1,1,1.000000,1.000000,15.842830,75.435250,1,0.000000,0.000000,0.000000,0.000000
3,4,479,20.091620,79.026615,2024-01-03,2024-01-04,2,2,2,2,0.259333,0.790000,0.96,303.415000,305.92,291.045000,291.20,Transient (1–2 active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=20.09162&m...,Industrial,Medium,Coal mine,none,none,none,Open-pit mine at event location; mapped mining...,"Yes - 500m, 1km",yes,2026-09-03,Exact thermal source unclear,0.790000,0.96,0.240416,303.415000,305.92,3.542605,291.045000,291.20,0.219203,2024-01-03,2024-01-04,2,2,2,1.000000,1.000000,20.091620,79.026615,2,0.259333,0.170000,2.505000,0.155000
4,5,368,27.558151,96.039266,2024-01-02,2024-01-27,19,26,22,19,0.482276,0.625455,1.09,301.573182,317.51,281.559545,283.52,Persistent (15+ active days),https://www.google.com/maps/@?api=1&map_action...,https://www.openstreetmap.org/?mlat=27.5581513...,Unknown,Low,none,none,none,Rural settlement/roads,Mixed rural landscape; no clear source identified,"Yes - 500m, 1km",yes,2026-09-03,Persistent activity but source unclear,0.625455,1.09,0.199469,301.573182,317.51,5.549614,281.559545,283.52,1.098698,2024-01-02,2024-01-27,19,22,26,0.730769,1.157895,27.558151,96.039266,19,0.482276,0.464545,15.936818,1.960455


In [18]:
behavior_features = [
    "mean_frp",
    "max_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5",
    "active_days",
    "duration_days",
    "activity_frequency",
    "detections_per_active_day",
    "spatial_diameter_km"
]

X_behavior = v11[behavior_features].copy()

scaler = RobustScaler()
X_behavior_scaled = scaler.fit_transform(X_behavior)

print("Behavior features:", len(behavior_features))

Behavior features: 11


In [19]:
behavior_features = [
    "mean_frp",
    "max_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5",
    "active_days",
    "duration_days",
    "activity_frequency",
    "detections_per_active_day",
    "spatial_diameter_km"
]

X_behavior = v11[behavior_features].copy()

scaler = RobustScaler()
X_behavior_scaled = scaler.fit_transform(X_behavior)

print("Behavior features:", len(behavior_features))

Behavior features: 11


In [20]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

v11["behavior_cluster"] = kmeans.fit_predict(
    X_behavior_scaled
)

cluster_profile = (
    v11.groupby("behavior_cluster")[behavior_features]
    .mean()
)

display(cluster_profile)

,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4,mean_bright_ti5,max_bright_ti5,active_days,duration_days,activity_frequency,detections_per_active_day,spatial_diameter_km
behavior_cluster,,,,,,,,,,,
0,1.327267,1.415353,304.966481,305.905420,283.354865,283.581200,1.226778,1.333547,0.979580,1.141791,0.063404
1,1.434003,2.995571,305.227772,319.426238,287.157875,290.732762,13.114286,16.366667,0.793454,2.260882,0.933761


In [21]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

v11["behavior_cluster"] = kmeans.fit_predict(
    X_behavior_scaled
)

cluster_profile = (
    v11.groupby("behavior_cluster")[behavior_features]
    .mean()
)

display(cluster_profile)

,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4,mean_bright_ti5,max_bright_ti5,active_days,duration_days,activity_frequency,detections_per_active_day,spatial_diameter_km
behavior_cluster,,,,,,,,,,,
0,1.327267,1.415353,304.966481,305.905420,283.354865,283.581200,1.226778,1.333547,0.979580,1.141791,0.063404
1,1.434003,2.995571,305.227772,319.426238,287.157875,290.732762,13.114286,16.366667,0.793454,2.260882,0.933761


In [22]:
cluster_activity = (
    v11.groupby("behavior_cluster")["active_days"]
    .mean()
)

persistent_cluster = cluster_activity.idxmax()

v11["behavior_label"] = np.where(
    v11["behavior_cluster"] == persistent_cluster,
    "Persistent",
    "Transient"
)

display(
    v11["behavior_label"]
    .value_counts()
)

behavior_label
Transient     4683
Persistent     210
Name: count, dtype: int64

In [23]:
iso = IsolationForest(
    contamination=0.05,
    random_state=42
)

v11["isolation_anomaly"] = (
    iso.fit_predict(X_behavior) == -1
)

v11["isolation_score"] = -iso.score_samples(
    X_behavior
)

print(
    "Isolation Forest anomalies:",
    v11["isolation_anomaly"].sum()
)

Isolation Forest anomalies: 245


In [24]:
iso = IsolationForest(
    contamination=0.05,
    random_state=42
)

v11["isolation_anomaly"] = (
    iso.fit_predict(X_behavior) == -1
)

v11["isolation_score"] = -iso.score_samples(
    X_behavior
)

print(
    "Isolation Forest anomalies:",
    v11["isolation_anomaly"].sum()
)

Isolation Forest anomalies: 245


In [25]:
model_outputs = v11[
    [
        "event_id",
        "behavior_cluster",
        "behavior_label",
        "isolation_anomaly",
        "isolation_score"
    ]
]

validation_analysis = validation_ml.merge(
    model_outputs,
    on="event_id",
    how="left"
)

display(
    validation_analysis[
        [
            "event_id",
            "actual_class",
            "confidence",
            "behavior_label",
            "isolation_anomaly"
        ]
    ]
)

,event_id,actual_class,confidence,behavior_label,isolation_anomaly
0,12,Industrial,High,Persistent,True
1,1347,Industrial,high,Persistent,True
2,1967,Unknown,low,Transient,False
3,479,Industrial,Medium,Transient,False
4,368,Unknown,Low,Persistent,True
5,3859,Unknown,Low,Transient,False
6,2201,Natural_Vegetation,medium,Transient,False
7,2882,Natural_Vegetation,high,Transient,False
8,172,Industrial,high,Persistent,True
9,693,Industrial,high,Transient,False


In [26]:
behavior_vs_actual = pd.crosstab(
    validation_analysis["actual_class"],
    validation_analysis["behavior_label"],
    margins=True
)

display(behavior_vs_actual)

behavior_label,Persistent,Transient,All
actual_class,,,
Industrial,5,3,8
Industrial,2,1,3
Natural_Vegetation,0,4,4
Unknown,1,2,3
All,8,10,18


In [27]:
anomaly_vs_actual = pd.crosstab(
    validation_analysis["actual_class"],
    validation_analysis["isolation_anomaly"],
    margins=True
)

display(anomaly_vs_actual)

isolation_anomaly,False,True,All
actual_class,,,
Industrial,3,5,8
Industrial,1,2,3
Natural_Vegetation,4,0,4
Unknown,2,1,3
All,10,8,18


In [28]:
class_behavior_profile = (
    validation_analysis
    .groupby("actual_class")[behavior_features]
    .mean()
    .round(3)
)

display(class_behavior_profile)

,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4,mean_bright_ti5,max_bright_ti5,active_days,duration_days,activity_frequency,detections_per_active_day,spatial_diameter_km
actual_class,,,,,,,,,,,
Industrial,1.332,2.640,304.983,315.269,287.708,291.054,11.625,13.500,0.905,2.558,0.858
Industrial,0.951,1.360,303.080,309.017,287.457,289.820,12.667,13.333,0.970,1.343,0.409
Natural_Vegetation,1.982,1.982,317.805,317.805,278.515,278.515,1.000,1.000,1.000,1.000,0.000
Unknown,1.130,1.563,306.926,315.027,288.169,289.387,7.333,9.667,0.910,1.553,0.356


In [29]:
confidence_summary = pd.crosstab(
    validation_analysis["actual_class"],
    validation_analysis["confidence"],
    margins=True
)

display(confidence_summary)

confidence,High,Low,Medium,high,low,medium,All
actual_class,,,,,,,
Industrial,0,0,2,5,0,1,8
Industrial,1,0,0,2,0,0,3
Natural_Vegetation,0,0,0,2,0,2,4
Unknown,0,2,0,0,1,0,3
All,1,2,2,9,1,3,18


In [30]:
binary_validation = validation_analysis[
    validation_analysis["actual_class"].isin(
        ["Industrial", "Agricultural", "Natural_Vegetation", "Other"]
    )
].copy()

binary_validation["actual_industrial"] = (
    binary_validation["actual_class"] == "Industrial"
)

display(
    binary_validation[
        [
            "event_id",
            "actual_class",
            "behavior_label",
            "isolation_anomaly"
        ]
    ]
)

,event_id,actual_class,behavior_label,isolation_anomaly
3,479,Industrial,Transient,False
6,2201,Natural_Vegetation,Transient,False
7,2882,Natural_Vegetation,Transient,False
8,172,Industrial,Persistent,True
10,3416,Natural_Vegetation,Transient,False
11,1181,Industrial,Transient,False
12,2074,Natural_Vegetation,Transient,False
13,1225,Industrial,Persistent,True
14,3906,Industrial,Transient,False
15,3393,Industrial,Persistent,True


In [31]:
industrial_profile = (
    binary_validation
    .groupby("actual_industrial")[behavior_features]
    .mean()
    .round(3)
)

display(industrial_profile)

,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4,mean_bright_ti5,max_bright_ti5,active_days,duration_days,activity_frequency,detections_per_active_day,spatial_diameter_km
actual_industrial,,,,,,,,,,,
False,1.982,1.982,317.805,317.805,278.515,278.515,1.000,1.0,1.000,1.000,0.000
True,1.332,2.640,304.983,315.269,287.708,291.054,11.625,13.5,0.905,2.558,0.858


In [32]:
industrial_behavior = pd.crosstab(
    binary_validation["actual_industrial"],
    binary_validation["behavior_label"],
    margins=True
)

display(industrial_behavior)

behavior_label,Persistent,Transient,All
actual_industrial,,,
False,0,4,4
True,5,3,8
All,5,7,12


In [34]:
thermal_features = [
    "mean_frp",
    "max_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5"
]

thermal_profile = (
    binary_validation
    .groupby("actual_class")[thermal_features]
    .mean()
    .round(3)
)

display(thermal_profile)

,mean_frp,max_frp,mean_bright_ti4,max_bright_ti4,mean_bright_ti5,max_bright_ti5
actual_class,,,,,,
Industrial,1.332,2.640,304.983,315.269,287.708,291.054
Natural_Vegetation,1.982,1.982,317.805,317.805,278.515,278.515


In [35]:
temporal_spatial_features = [
    "active_days",
    "duration_days",
    "detection_count",
    "daily_object_count",
    "detections_per_active_day",
    "spatial_diameter_km"
]

profile = (
    binary_validation
    .groupby("actual_class")[temporal_spatial_features]
    .mean()
    .round(3)
)

display(profile)

,active_days,duration_days,detection_count,daily_object_count,detections_per_active_day,spatial_diameter_km
actual_class,,,,,,
Industrial,11.625,13.5,32.375,27.375,2.558,0.858
Natural_Vegetation,1.000,1.0,1.000,1.000,1.000,0.000


In [36]:
display(
    validation_analysis[
        [
            "event_id",
            "actual_class",
            "confidence",
            "behavior_label",
            "isolation_anomaly",
            "mean_frp",
            "max_frp",
            "active_days",
            "duration_days",
            "spatial_diameter_km"
        ]
    ].sort_values("event_id")
)

,event_id,actual_class,confidence,behavior_label,isolation_anomaly,mean_frp,max_frp,active_days,duration_days,spatial_diameter_km
0,12,Industrial,High,Persistent,True,0.973846,1.51,17,17,0.539481
16,100,Industrial,Medium,Persistent,True,1.862083,4.80,26,31,1.409461
8,172,Industrial,high,Persistent,True,1.443429,2.83,16,16,1.185706
17,341,Industrial,high,Persistent,True,1.697059,3.52,20,24,0.887156
4,368,Unknown,Low,Persistent,True,0.625455,1.09,19,26,0.482276
3,479,Industrial,Medium,Transient,False,0.790000,0.96,2,2,0.259333
9,693,Industrial,high,Transient,False,0.660000,0.66,1,1,0.000000
11,1181,Industrial,high,Transient,False,1.040000,1.04,1,1,0.000000
13,1225,Industrial,high,Persistent,True,1.059375,2.31,20,23,1.106945
1,1347,Industrial,high,Persistent,True,1.218333,1.91,20,22,0.688072


In [37]:
OUTPUT_FILE = "viirs_validation_analysis_v1.csv"

validation_analysis.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)

Saved: viirs_validation_analysis_v1.csv


In [38]:
# ============================================================
# CELL 30 — LOAD V11 EVENT DATA + OSM CONTEXT
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.neighbors import BallTree

EARTH_RADIUS_KM = 6371.0088

V11_FILE = "viirs_v11_final_event_dataset.csv"
OSM_FILE = "osm_points_v7.csv"

events_v11 = pd.read_csv(V11_FILE)
osm_points = pd.read_csv(OSM_FILE)

print("V11 events:", events_v11.shape)
print("OSM points:", osm_points.shape)

print("\nV11 columns:")
print(events_v11.columns.tolist())

print("\nOSM columns:")
print(osm_points.columns.tolist())

V11 events: (4893, 24)
OSM points: (1981, 15)

V11 columns:
['event_id', 'mean_frp', 'max_frp', 'std_frp', 'mean_bright_ti4', 'max_bright_ti4', 'std_bright_ti4', 'mean_bright_ti5', 'max_bright_ti5', 'std_bright_ti5', 'start_date', 'end_date', 'active_days', 'detection_count', 'duration_days', 'activity_frequency', 'detections_per_active_day', 'centroid_lat', 'centroid_lon', 'daily_object_count', 'spatial_diameter_km', 'frp_range', 'ti4_range', 'ti5_range']

OSM columns:
['osm_type', 'osm_id', 'latitude', 'longitude', 'industrial', 'landuse', 'power', 'man_made', 'building', 'product', 'plant_source', 'plant_method', 'resource', 'description', 'source_file']


In [39]:
# ============================================================
# CELL 31 — NORMALIZE OSM ENTITY TYPES
# ============================================================

osm = osm_points.copy()

for col in [
    "industrial",
    "landuse",
    "power",
    "man_made",
    "building",
    "product",
    "plant_source",
    "plant_method",
    "resource",
    "description"
]:
    if col in osm.columns:
        osm[col] = (
            osm[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )

# ------------------------------------------------------------
# DOMAIN ENTITY FLAGS
# ------------------------------------------------------------

osm["entity_industrial_zone"] = (
    osm["landuse"] == "industrial"
)

osm["entity_factory"] = osm["industrial"].isin([
    "factory",
    "concrete_plant",
    "machine_shop",
    "food_industry",
    "biotechnology company",
    "biotechnology_company",
    "pharmaceutical company",
    "pharmaceutical_company",
    "research company",
    "research institute",
    "laboratory",
    "agrochemical company",
    "refractory_supplier",
    "oil",
    "oil_mill",
    "rice_mill",
    "grinding_mill",
    "sawmill",
    "mill"
])

osm["entity_mine"] = (
    (osm["industrial"] == "mine") |
    (osm["landuse"] == "quarry")
)

osm["entity_brick"] = (
    osm["industrial"].isin([
        "brickyard",
        "brickworks"
    ])
    |
    (osm["man_made"] == "kiln")
)

osm["entity_works"] = (
    osm["man_made"] == "works"
)

osm["entity_depot"] = osm["industrial"].isin([
    "depot",
    "bus_depot"
])

osm["entity_power"] = (
    osm["power"] == "plant"
)

osm["entity_other_industry"] = osm["industrial"].isin([
    "slaughterhouse",
    "scrap_yard",
    "warehouse",
    "port",
    "cooling",
    "distributor",
    "business"
])

entity_cols = [
    "entity_industrial_zone",
    "entity_factory",
    "entity_mine",
    "entity_brick",
    "entity_works",
    "entity_depot",
    "entity_power",
    "entity_other_industry"
]

osm["entity_any_industry"] = osm[entity_cols].any(axis=1)

print("OSM entity counts:")
print(
    osm[entity_cols]
    .sum()
    .sort_values(ascending=False)
)

OSM entity counts:
entity_works              1563
entity_industrial_zone     360
entity_other_industry      171
entity_factory              67
entity_depot                39
entity_brick                 6
entity_mine                  1
entity_power                 0
dtype: int64


In [40]:
# ============================================================
# CELL 32 — OSM PROXIMITY FEATURES
# ============================================================

event_coords = np.radians(
    events_v11[
        ["centroid_lat", "centroid_lon"]
    ].values
)

osm_coords = np.radians(
    osm[
        ["latitude", "longitude"]
    ].values
)

entity_trees = {}

for entity in entity_cols:

    mask = osm[entity].values

    if mask.sum() > 0:

        entity_trees[entity] = BallTree(
            osm_coords[mask],
            metric="haversine"
        )

radii_km = {
    "375m": 0.375,
    "1km": 1.0,
    "3km": 3.0
}

osm_features = pd.DataFrame({
    "event_id": events_v11["event_id"]
})

for entity in entity_cols:

    prefix = entity.replace("entity_", "")

    if entity not in entity_trees:

        osm_features[
            f"nearest_{prefix}_km"
        ] = np.nan

        for radius_name in radii_km:

            osm_features[
                f"{prefix}_count_{radius_name}"
            ] = 0

        continue

    tree = entity_trees[entity]

    # Nearest entity distance
    dist, _ = tree.query(
        event_coords,
        k=1
    )

    osm_features[
        f"nearest_{prefix}_km"
    ] = (
        dist[:, 0] *
        EARTH_RADIUS_KM
    )

    # Entity counts
    for radius_name, radius_km in radii_km.items():

        counts = tree.query_radius(
            event_coords,
            r=radius_km / EARTH_RADIUS_KM,
            count_only=True
        )

        osm_features[
            f"{prefix}_count_{radius_name}"
        ] = counts


events_geo = events_v11.merge(
    osm_features,
    on="event_id",
    how="left"
)

print("Contextual event table:", events_geo.shape)

print("\nOSM features added:")
print([
    c for c in events_geo.columns
    if c not in events_v11.columns
])

Contextual event table: (4893, 56)

OSM features added:
['nearest_industrial_zone_km', 'industrial_zone_count_375m', 'industrial_zone_count_1km', 'industrial_zone_count_3km', 'nearest_factory_km', 'factory_count_375m', 'factory_count_1km', 'factory_count_3km', 'nearest_mine_km', 'mine_count_375m', 'mine_count_1km', 'mine_count_3km', 'nearest_brick_km', 'brick_count_375m', 'brick_count_1km', 'brick_count_3km', 'nearest_works_km', 'works_count_375m', 'works_count_1km', 'works_count_3km', 'nearest_depot_km', 'depot_count_375m', 'depot_count_1km', 'depot_count_3km', 'nearest_power_km', 'power_count_375m', 'power_count_1km', 'power_count_3km', 'nearest_other_industry_km', 'other_industry_count_375m', 'other_industry_count_1km', 'other_industry_count_3km']


In [41]:
# ============================================================
# CELL 33 — OSM COVERAGE VALIDATION
# ============================================================

osm_count_cols = [
    c for c in events_geo.columns
    if "_count_" in c
]

print("OSM COUNT COVERAGE")
print("=" * 50)

for radius in ["375m", "1km", "3km"]:

    print(f"\n--- {radius} ---")

    radius_cols = [
        c for c in osm_count_cols
        if c.endswith(radius)
    ]

    for col in radius_cols:

        n = (
            events_geo[col] > 0
        ).sum()

        print(
            f"{col:45s} "
            f"{n:4d} events "
            f"({n / len(events_geo) * 100:.2f}%)"
        )

# Any OSM context
events_geo["has_osm_context"] = (
    events_geo[osm_count_cols]
    .gt(0)
    .any(axis=1)
)

print("\nEvents with ANY OSM context:")
print(
    events_geo["has_osm_context"].sum(),
    "/",
    len(events_geo)
)

print(
    "Coverage:",
    round(
        events_geo["has_osm_context"].mean() * 100,
        2
    ),
    "%"
)

OSM COUNT COVERAGE

--- 375m ---
industrial_zone_count_375m                       0 events (0.00%)
factory_count_375m                               0 events (0.00%)
mine_count_375m                                  0 events (0.00%)
brick_count_375m                                 0 events (0.00%)
works_count_375m                                 6 events (0.12%)
depot_count_375m                                 0 events (0.00%)
power_count_375m                                 0 events (0.00%)
other_industry_count_375m                        0 events (0.00%)

--- 1km ---
industrial_zone_count_1km                        0 events (0.00%)
factory_count_1km                                0 events (0.00%)
mine_count_1km                                   0 events (0.00%)
brick_count_1km                                  0 events (0.00%)
works_count_1km                                 21 events (0.43%)
depot_count_1km                                  0 events (0.00%)
power_count_1km               

In [42]:
# ============================================================
# CELL 34 — MERGE VALIDATION LABELS
# ============================================================

# validation dataframe already exists
# from your previous notebook cells

validation_labels = validation[
    [
        "event_id",
        "actual_class",
        "confidence",
        "industrial_evidence",
        "agricultural_evidence",
        "natural_evidence",
        "other_evidence",
        "evidence_notes"
    ]
].copy()

validation_labels["event_id"] = (
    pd.to_numeric(
        validation_labels["event_id"],
        errors="coerce"
    )
)

events_geo["event_id"] = (
    pd.to_numeric(
        events_geo["event_id"],
        errors="coerce"
    )
)

validation_geo = events_geo.merge(
    validation_labels,
    on="event_id",
    how="inner"
)

print("Validation + geographic context:")
print(validation_geo.shape)

print("\nActual classes:")
print(
    validation_geo["actual_class"]
    .value_counts(dropna=False)
)

Validation + geographic context:
(60, 64)

Actual classes:
actual_class
NaN                   42
Industrial             8
Natural_Vegetation     4
Industrial             3
Unknown                3
Name: count, dtype: int64


In [43]:
# ============================================================
# CELL 35 — BEHAVIOR × OSM CONTEXT
# ============================================================

# Get behavioral results from your existing validation notebook
# The previous notebook already produced these.

behavior_cols = [
    "event_id",
    "behavior_label",
    "isolation_anomaly",
    "isolation_score"
]

available_behavior_cols = [
    c for c in behavior_cols
    if c in validation_analysis.columns
]

behavior_results = (
    validation_analysis[
        available_behavior_cols
    ]
    .drop_duplicates("event_id")
)

validation_geo = validation_geo.merge(
    behavior_results,
    on="event_id",
    how="left"
)

print("Validation rows:", len(validation_geo))

print("\nBehavior groups:")
print(
    validation_geo["behavior_label"]
    .value_counts(dropna=False)
)

print("\nBehavior × Actual:")
display(
    pd.crosstab(
        validation_geo["behavior_label"],
        validation_geo["actual_class"]
    )
)

Validation rows: 60

Behavior groups:
behavior_label
NaN           42
Transient     10
Persistent     8
Name: count, dtype: int64

Behavior × Actual:


actual_class,Industrial,Industrial,Natural_Vegetation,Unknown
behavior_label,,,,
Persistent,5,2,0,1
Transient,3,1,4,2


In [44]:
# ============================================================
# CELL 36 — OSM CONTEXT vs ACTUAL CLASS
# ============================================================

context_cols = [
    "entity_industrial_zone_count_3km",
    "entity_factory_count_3km",
    "entity_mine_count_3km",
    "entity_brick_count_3km",
    "entity_works_count_3km",
    "entity_depot_count_3km",
    "entity_power_count_3km",
    "entity_other_industry_count_3km"
]

context_cols = [
    c for c in context_cols
    if c in validation_geo.columns
]

print("Mean OSM context by actual class:")

display(
    validation_geo
    .groupby("actual_class")[context_cols]
    .mean()
    .round(3)
)

Mean OSM context by actual class:


""
actual_class
Industrial
Industrial
Natural_Vegetation
Unknown


In [45]:
# ============================================================
# CELL 37 — OSM PRESENCE vs ACTUAL CLASS
# ============================================================

for col in context_cols:

    name = col.replace(
        "_count_3km",
        ""
    )

    validation_geo[
        f"has_{name}_3km"
    ] = (
        validation_geo[col] > 0
    )

presence_cols = [
    c for c in validation_geo.columns
    if c.startswith("has_")
]

print("OSM evidence presence by actual class:")

display(
    validation_geo
    .groupby("actual_class")[presence_cols]
    .mean()
    .mul(100)
    .round(2)
)

OSM evidence presence by actual class:


,has_osm_context
actual_class,
Industrial,0.00
Industrial,0.00
Natural_Vegetation,0.00
Unknown,33.33


In [46]:
# ============================================================
# CELL 38 — OSM DISTANCE vs ACTUAL CLASS
# ============================================================

nearest_cols = [
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_mine_km",
    "nearest_brick_km",
    "nearest_works_km",
    "nearest_depot_km",
    "nearest_power_km",
    "nearest_other_industry_km"
]

nearest_cols = [
    c for c in nearest_cols
    if c in validation_geo.columns
]

print("OSM distance by actual class:")

display(
    validation_geo
    .groupby("actual_class")[nearest_cols]
    .agg(
        ["count", "median", "mean", "min"]
    )
    .round(3)
)

OSM distance by actual class:


nearest_industrial_zone_km                             \
                                        count   median     mean      min   
actual_class                                                               
Industrial                                  8  144.933  154.071   44.016   
Industrial                                  3  132.597  161.855   50.007   
Natural_Vegetation                          4  340.474  335.185  149.220   
Unknown                                     3   64.214   83.076    3.812   

                   nearest_factory_km                             \
                                count   median     mean      min   
actual_class                                                       
Industrial                          8  283.616  293.615  157.768   
Industrial                          3  404.604  387.949  132.781   
Natural_Vegetation                  4  438.088  431.707  252.452   
Unknown                             3  393.169  635.350  147.738   

                   nearest_mine_km                                \
                             count    median      mean       min   
actual_class                                                       
Industrial                       8   857.068   990.336   407.627   
Industrial                       3  1542.121  1392.808   619.174   
Natural_Vegetation               4  1517.660  1530.818  1343.179   
Unknown                          3  1853.244  1537.302   649.287   

                   nearest_brick_km                             \
                              count   median     mean      min   
actual_class                                                     
Industrial                        8  597.701  594.013  301.233   
Industrial                        3  634.629  609.016  529.438   
Natural_Vegetation                4  290.660  282.475  143.603   
Unknown                           3  623.533  954.606  362.145   

                   nearest_works_km                         nearest_depot_km  \
                              count  median    mean     min            count   
actual_class                                                                   
Industrial                        8  55.976  48.585   7.647                8   
Industrial                        3  88.086  80.206  24.224                3   
Natural_Vegetation                4  69.844  66.505  26.448                4   
Unknown                           3  92.680  63.002   2.854                3   

                                              nearest_power_km              \
                     median     mean      min            count median mean   
actual_class                                                                 
Industrial          451.876  439.185  132.188                0    NaN  NaN   
Industrial          410.790  382.357  260.604                0    NaN  NaN   
Natural_Vegetation  432.603  425.672  248.958                0    NaN  NaN   
Unknown              64.214  131.251    3.812                0    NaN  NaN   

                       nearest_other_industry_km                             
                   min                     count   median     mean      min  
actual_class                                                                 
Industrial         NaN                         8  244.247  265.127  120.990  
Industrial         NaN                         3  273.486  208.818   50.007  
Natural_Vegetation NaN                         4  487.135  479.858  305.077  
Unknown            NaN                         3  181.200  448.227   57.337

In [47]:
# ============================================================
# CELL 39 — WORLDCOVER CENTROID CONTEXT
# ============================================================

import glob
import rasterio

WORLD_COVER_DIR = "worldcover_india"

worldcover_files = sorted(
    glob.glob(
        os.path.join(
            WORLD_COVER_DIR,
            "ESA_WorldCover_10m_2021_v200_*_Map.tif"
        )
    )
)

print(
    "WorldCover files found:",
    len(worldcover_files)
)

tiles = []

for path in worldcover_files:

    try:

        with rasterio.open(path) as src:

            bounds = src.bounds

            tiles.append({
                "path": path,
                "min_lon": bounds.left,
                "max_lon": bounds.right,
                "min_lat": bounds.bottom,
                "max_lat": bounds.top
            })

    except Exception:

        pass

print(
    "Valid WorldCover tiles:",
    len(tiles)
)

wc_class = np.full(
    len(validation_geo),
    np.nan
)

for tile in tiles:

    mask = (
        (validation_geo["centroid_lon"] >= tile["min_lon"]) &
        (validation_geo["centroid_lon"] < tile["max_lon"]) &
        (validation_geo["centroid_lat"] >= tile["min_lat"]) &
        (validation_geo["centroid_lat"] < tile["max_lat"]) &
        np.isnan(wc_class)
    )

    indices = np.where(mask)[0]

    if len(indices) == 0:
        continue

    try:

        with rasterio.open(tile["path"]) as src:

            coords = [
                (
                    validation_geo.iloc[i]["centroid_lon"],
                    validation_geo.iloc[i]["centroid_lat"]
                )
                for i in indices
            ]

            values = list(
                src.sample(coords)
            )

            for i, value in zip(
                indices,
                values
            ):
                wc_class[i] = value[0]

    except Exception:

        pass

validation_geo["worldcover_class"] = wc_class

print("\nWorldCover coverage:")
print(
    validation_geo["worldcover_class"]
    .notna()
    .sum(),
    "/",
    len(validation_geo)
)

WorldCover files found: 102
Valid WorldCover tiles: 102

WorldCover coverage:
57 / 60


In [48]:
# ============================================================
# CELL 40 — WORLDCOVER SEMANTIC FEATURES
# ============================================================

WC_FEATURES = {
    10: "tree_cover",
    20: "shrubland",
    30: "grassland",
    40: "cropland",
    50: "built_up",
    60: "bare_sparse",
    70: "snow_ice",
    80: "permanent_water",
    90: "wetland",
    95: "mangroves",
    100: "moss_lichen"
}

validation_geo["worldcover_missing"] = (
    validation_geo["worldcover_class"]
    .isna()
    .astype(int)
)

for code, name in WC_FEATURES.items():

    validation_geo[
        f"wc_{name}"
    ] = (
        validation_geo["worldcover_class"]
        == code
    ).astype(int)

print("WorldCover distribution:")

display(
    validation_geo[
        [
            c for c in validation_geo.columns
            if c.startswith("wc_")
        ]
    ]
    .sum()
    .sort_values(ascending=False)
)

WorldCover distribution:


wc_bare_sparse        19
wc_tree_cover         13
wc_grassland           9
wc_built_up            9
wc_cropland            6
wc_permanent_water     1
wc_shrubland           0
wc_snow_ice            0
wc_wetland             0
wc_mangroves           0
wc_moss_lichen         0
dtype: int64

In [49]:
# ============================================================
# CELL 41 — WORLDCOVER vs ACTUAL CLASS
# ============================================================

wc_cols = [
    "wc_tree_cover",
    "wc_shrubland",
    "wc_grassland",
    "wc_cropland",
    "wc_built_up",
    "wc_bare_sparse",
    "wc_permanent_water"
]

wc_cols = [
    c for c in wc_cols
    if c in validation_geo.columns
]

print("WorldCover context by actual class:")

display(
    validation_geo
    .groupby("actual_class")[wc_cols]
    .mean()
    .mul(100)
    .round(2)
)

WorldCover context by actual class:


,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_permanent_water
actual_class,,,,,,,
Industrial,12.50,0.0,0.0,0.00,25.00,62.50,0.0
Industrial,33.33,0.0,0.0,0.00,33.33,33.33,0.0
Natural_Vegetation,25.00,0.0,25.0,0.00,0.00,0.00,0.0
Unknown,33.33,0.0,0.0,66.67,0.00,0.00,0.0


In [50]:
# ============================================================
# CELL 42 — BEHAVIOR × GEOGRAPHIC CONTEXT
# ============================================================

context_analysis_cols = (
    context_cols +
    wc_cols
)

context_analysis_cols = [
    c for c in context_analysis_cols
    if c in validation_geo.columns
]

print("Context by behavioral group:")

display(
    validation_geo
    .groupby("behavior_label")[context_analysis_cols]
    .mean()
    .round(3)
)

print("\nBehavior × Actual Class:")

display(
    pd.crosstab(
        validation_geo["behavior_label"],
        validation_geo["actual_class"]
    )
)

Context by behavioral group:


,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_permanent_water
behavior_label,,,,,,,
Persistent,0.125,0.0,0.0,0.125,0.25,0.5,0.0
Transient,0.300,0.0,0.1,0.100,0.10,0.2,0.0



Behavior × Actual Class:


actual_class,Industrial,Industrial,Natural_Vegetation,Unknown
behavior_label,,,,
Persistent,5,2,0,1
Transient,3,1,4,2


In [52]:
# ============================================================
# CELL 43 — INDUSTRIAL vs NON-INDUSTRIAL
# ============================================================

validation_geo["actual_binary"] = pd.Series(
    pd.NA,
    index=validation_geo.index,
    dtype="string"
)

validation_geo.loc[
    validation_geo["actual_class"] == "Industrial",
    "actual_binary"
] = "Industrial"

validation_geo.loc[
    validation_geo["actual_class"].isin([
        "Agricultural",
        "Natural_Vegetation",
        "Other"
    ]),
    "actual_binary"
] = "Non_Industrial"

binary_df = validation_geo[
    validation_geo["actual_binary"].notna()
].copy()

print("Binary validation classes:")
print(
    binary_df["actual_binary"]
    .value_counts()
)

print("\nExcluded from binary validation:")
print(
    validation_geo[
        validation_geo["actual_binary"].isna()
    ]["actual_class"]
    .value_counts(dropna=False)
)

print("\nBehavior distribution:")

display(
    pd.crosstab(
        binary_df["actual_binary"],
        binary_df["behavior_label"],
        normalize="index"
    ).mul(100).round(2)
)

Binary validation classes:
actual_binary
Industrial        8
Non_Industrial    4
Name: count, dtype: Int64

Excluded from binary validation:
actual_class
NaN            42
Industrial      3
Unknown         3
Name: count, dtype: int64

Behavior distribution:


behavior_label,Persistent,Transient
actual_binary,,
Industrial,62.5,37.5
Non_Industrial,0.0,100.0


In [53]:
# ============================================================
# CELL 44 — INDUSTRIAL vs OSM EVIDENCE
# ============================================================

print("OSM context by binary actual class:")

display(
    binary_df
    .groupby("actual_binary")[context_cols]
    .mean()
    .round(3)
)

print("\nOSM presence by binary actual class:")

display(
    binary_df
    .groupby("actual_binary")[
        [
            c for c in presence_cols
            if c in binary_df.columns
        ]
    ]
    .mean()
    .mul(100)
    .round(2)
)

OSM context by binary actual class:


""
actual_binary
Industrial
Non_Industrial



OSM presence by binary actual class:


,has_osm_context
actual_binary,
Industrial,0.0
Non_Industrial,0.0


In [54]:
# ============================================================
# CELL 45 — INDUSTRIAL vs WORLDCOVER
# ============================================================

print("WorldCover by binary actual class:")

display(
    binary_df
    .groupby("actual_binary")[wc_cols]
    .mean()
    .mul(100)
    .round(2)
)

WorldCover by binary actual class:


,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_permanent_water
actual_binary,,,,,,,
Industrial,12.5,0.0,0.0,0.0,25.0,62.5,0.0
Non_Industrial,25.0,0.0,25.0,0.0,0.0,0.0,0.0


In [55]:
# ============================================================
# CELL 46 — COMBINED EVIDENCE TABLE
# ============================================================

evidence_cols = [
    "event_id",
    "actual_class",
    "actual_binary",
    "confidence",
    "behavior_label",
    "isolation_anomaly",
    "isolation_score",

    "mean_frp",
    "max_frp",
    "active_days",
    "duration_days",
    "detection_count",
    "spatial_diameter_km"
]

evidence_cols += [
    c for c in nearest_cols
    if c in binary_df.columns
]

evidence_cols += [
    c for c in context_cols
    if c in binary_df.columns
]

evidence_cols += [
    c for c in wc_cols
    if c in binary_df.columns
]

evidence_cols = list(dict.fromkeys(evidence_cols))

evidence_table = binary_df[
    evidence_cols
].copy()

print(
    "Evidence table:",
    evidence_table.shape
)

display(
    evidence_table.head(20)
)

Evidence table: (12, 28)


,event_id,actual_class,actual_binary,confidence,behavior_label,isolation_anomaly,isolation_score,mean_frp,max_frp,active_days,duration_days,detection_count,spatial_diameter_km,nearest_industrial_zone_km,nearest_factory_km,nearest_mine_km,nearest_brick_km,nearest_works_km,nearest_depot_km,nearest_power_km,nearest_other_industry_km,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_permanent_water
6,100,Industrial,Industrial,Medium,Persistent,True,0.714248,1.862083,4.80,26,31,96,1.409461,44.015799,319.303903,407.627108,888.746632,7.647401,599.964016,NaN,120.990298,0,0,0,0,0,1,0
9,172,Industrial,Industrial,high,Persistent,True,0.629462,1.443429,2.83,16,16,35,1.185706,129.262480,234.167092,498.076911,784.752536,12.632172,594.372363,NaN,188.791237,1,0,0,0,0,0,0
16,341,Industrial,Industrial,high,Persistent,True,0.661650,1.697059,3.52,20,24,34,0.887156,247.910511,477.836281,1535.161130,628.986765,87.807182,495.198331,NaN,421.261048,0,0,0,0,1,0,0
18,479,Industrial,Industrial,Medium,Transient,False,0.409055,0.790000,0.96,2,2,2,0.259333,132.188123,272.836367,1270.549571,301.233213,16.428585,132.188123,NaN,258.272316,0,0,0,0,0,1,0
26,1181,Industrial,Industrial,high,Transient,False,0.368037,1.040000,1.04,1,1,1,0.000000,191.334864,340.754117,1787.911040,309.779307,66.524461,229.467075,NaN,230.222527,0,0,0,0,1,0,0
30,1225,Industrial,Industrial,high,Persistent,True,0.650770,1.059375,2.31,20,23,32,1.106945,127.948861,294.394900,751.992821,794.975011,52.145422,683.506881,NaN,127.948861,0,0,0,0,0,1,0
36,2074,Natural_Vegetation,Non_Industrial,medium,Transient,False,0.435743,2.130000,2.13,1,1,1,0.000000,205.112865,306.791188,1343.178887,202.143128,79.464671,304.490844,NaN,359.718661,0,0,1,0,0,0,0
37,2201,Natural_Vegetation,Non_Industrial,medium,Transient,False,0.494796,2.360000,2.36,1,1,1,0.000000,475.835336,569.385115,1683.270874,379.177750,60.223701,560.714516,NaN,614.551357,0,0,0,0,0,0,0
43,2882,Natural_Vegetation,Non_Industrial,high,Transient,False,0.377118,1.720000,1.72,1,1,1,0.000000,149.220109,252.452289,1352.049612,143.603404,26.447652,248.958387,NaN,305.076889,1,0,0,0,0,0,0
47,3393,Industrial,Industrial,high,Persistent,True,0.702604,2.091724,4.99,7,10,58,2.019369,157.676951,157.768035,709.227297,566.416215,59.807534,370.232383,NaN,366.066901,0,0,0,0,0,1,0


In [56]:
# ============================================================
# CELL 47 — FINAL VALIDATION EVIDENCE COMPARISON
# ============================================================

comparison_features = [
    "mean_frp",
    "max_frp",
    "active_days",
    "duration_days",
    "detection_count",
    "spatial_diameter_km",

    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_mine_km",
    "nearest_brick_km",
    "nearest_works_km",

    "entity_industrial_zone_count_3km",
    "entity_factory_count_3km",
    "entity_mine_count_3km",
    "entity_brick_count_3km",
    "entity_works_count_3km",

    "wc_tree_cover",
    "wc_cropland",
    "wc_built_up",
    "wc_bare_sparse"
]

comparison_features = [
    c for c in comparison_features
    if c in binary_df.columns
]

display(
    binary_df
    .groupby("actual_binary")[comparison_features]
    .agg(["count", "median", "mean"])
    .round(3)
)

mean_frp               max_frp               active_days  \
                  count median   mean   count median   mean       count   
actual_binary                                                             
Industrial            8  1.251  1.332       8  2.570  2.640           8   
Non_Industrial        4  1.925  1.982       4  1.925  1.982           4   

                              duration_days              detection_count  \
               median    mean         count median  mean           count   
actual_binary                                                              
Industrial       11.5  11.625             8   13.0  13.5               8   
Non_Industrial    1.0   1.000             4    1.0   1.0               4   

                              spatial_diameter_km                \
               median    mean               count median   mean   
actual_binary                                                     
Industrial       33.0  32.375                   8  0.997  0.858   
Non_Industrial    1.0   1.000                   4  0.000  0.000   

               nearest_industrial_zone_km                    \
                                    count   median     mean   
actual_binary                                                 
Industrial                              8  144.933  154.071   
Non_Industrial                          4  340.474  335.185   

               nearest_factory_km                   nearest_mine_km            \
                            count   median     mean           count    median   
actual_binary                                                                   
Industrial                      8  283.616  293.615               8   857.068   
Non_Industrial                  4  438.088  431.707               4  1517.660   

                         nearest_brick_km                   nearest_works_km  \
                    mean            count   median     mean            count   
actual_binary                                                                  
Industrial       990.336                8  597.701  594.013                8   
Non_Industrial  1530.818                4  290.660  282.475                4   

                               wc_tree_cover               wc_cropland         \
                median    mean         count median   mean       count median   
actual_binary                                                                   
Industrial      55.976  48.585             8    0.0  0.125           8    0.0   
Non_Industrial  69.844  66.505             4    0.0  0.250           4    0.0   

                    wc_built_up              wc_bare_sparse                
               mean       count median  mean          count median   mean  
actual_binary                                                              
Industrial      0.0           8    0.0  0.25              8    1.0  0.625  
Non_Industrial  0.0           4    0.0  0.00              4    0.0  0.000

In [57]:
# ============================================================
# CELL 48 — EVENT-LEVEL VALIDATION REVIEW
# ============================================================

review_cols = [
    "event_id",
    "actual_class",
    "confidence",
    "behavior_label",
    "isolation_anomaly",
    "active_days",
    "duration_days",
    "mean_frp",
    "max_frp",
    "spatial_diameter_km",
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_mine_km",
    "nearest_works_km",
    "entity_works_count_3km",
    "entity_industrial_zone_count_3km",
    "wc_built_up",
    "wc_cropland",
    "wc_tree_cover",
    "wc_grassland",
    "wc_bare_sparse",
    "evidence_notes"
]

review_cols = [
    c for c in review_cols
    if c in validation_geo.columns
]

validation_review = (
    validation_geo[review_cols]
    .sort_values(
        ["actual_class", "event_id"]
    )
)

display(validation_review)

,event_id,actual_class,confidence,behavior_label,isolation_anomaly,active_days,duration_days,mean_frp,max_frp,spatial_diameter_km,nearest_industrial_zone_km,nearest_factory_km,nearest_mine_km,nearest_works_km,wc_built_up,wc_cropland,wc_tree_cover,wc_grassland,wc_bare_sparse,evidence_notes
6,100,Industrial,Medium,Persistent,True,26,31,1.862083,4.80,1.409461,44.015799,319.303903,407.627108,7.647401,0,0,0,0,1,Event lies within a large open-pit mining land...
9,172,Industrial,high,Persistent,True,16,16,1.443429,2.83,1.185706,129.262480,234.167092,498.076911,12.632172,0,0,1,0,0,Persistent event within a large steel industri...
16,341,Industrial,high,Persistent,True,20,24,1.697059,3.52,0.887156,247.910511,477.836281,1535.161130,87.807182,1,0,0,0,0,Event is located within a clearly visible ceme...
18,479,Industrial,Medium,Transient,False,2,2,0.790000,0.96,0.259333,132.188123,272.836367,1270.549571,16.428585,0,0,0,0,1,Open-pit mine at event location; mapped mining...
26,1181,Industrial,high,Transient,False,1,1,1.040000,1.04,0.000000,191.334864,340.754117,1787.911040,66.524461,1,0,0,0,0,Event lies within a large industrial complex; ...
30,1225,Industrial,high,Persistent,True,20,23,1.059375,2.31,1.106945,127.948861,294.394900,751.992821,52.145422,0,0,0,0,1,Persistent event within an open-pit mining are...
47,3393,Industrial,high,Persistent,True,7,10,2.091724,4.99,2.019369,157.676951,157.768035,709.227297,59.807534,0,0,0,0,1,Event is located within a large open-pit coal ...
55,3906,Industrial,medium,Transient,False,1,1,0.670000,0.67,0.000000,202.228141,251.857858,962.144029,85.685083,0,0,0,0,1,Event lies in a large disturbed industrial/ext...
0,12,Industrial,High,Persistent,True,17,17,0.973846,1.51,0.539481,132.597362,132.780696,619.174361,128.310010,0,0,0,0,1,Industrial complex; persistent activity
19,693,Industrial,high,Transient,False,1,1,0.660000,0.66,0.000000,50.006706,404.604189,2017.129658,24.223901,0,0,1,0,0,Event is located within a large industrial are...


In [58]:
# ============================================================
# CELL 49 — SAVE CONTEXTUAL VALIDATION DATASET
# ============================================================

OUTPUT_FILE = (
    "viirs_validation_context_analysis_v1.csv"
)

validation_geo.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)
print("Shape:", validation_geo.shape)

Saved: viirs_validation_context_analysis_v1.csv
Shape: (60, 81)


In [59]:
# ============================================================
# CELL 50 — VALIDATION LABEL DIAGNOSTIC
# ============================================================

print("RAW actual_class values:")
print(
    validation_geo["actual_class"]
    .value_counts(dropna=False)
)

print("\nRAW actual_class representations:")

for value in validation_geo["actual_class"].dropna().unique():
    print(
        repr(value),
        "->",
        str(value).strip()
    )

print("\nRAW confidence values:")

for value in validation_geo["confidence"].dropna().unique():
    print(repr(value))

RAW actual_class values:
actual_class
NaN                   42
Industrial             8
Natural_Vegetation     4
Industrial             3
Unknown                3
Name: count, dtype: int64

RAW actual_class representations:
'Industrial ' -> Industrial
'Industrial' -> Industrial
'Unknown' -> Unknown
'Natural_Vegetation' -> Natural_Vegetation

RAW confidence values:
'High'
'Medium'
'high'
'Low'
'low'
'medium'


In [60]:
# ============================================================
# CELL 51 — NORMALIZE VALIDATION LABELS
# ============================================================

validation_geo["actual_class_clean"] = (
    validation_geo["actual_class"]
    .astype("string")
    .str.strip()
)

validation_geo["actual_binary"] = pd.Series(
    pd.NA,
    index=validation_geo.index,
    dtype="string"
)

validation_geo.loc[
    validation_geo["actual_class_clean"] == "Industrial",
    "actual_binary"
] = "Industrial"

validation_geo.loc[
    validation_geo["actual_class_clean"].isin([
        "Agricultural",
        "Natural_Vegetation",
        "Other"
    ]),
    "actual_binary"
] = "Non_Industrial"

print("Clean class distribution:")
print(
    validation_geo["actual_class_clean"]
    .value_counts(dropna=False)
)

print("\nBinary validation:")
print(
    validation_geo["actual_binary"]
    .value_counts(dropna=False)
)

Clean class distribution:
actual_class_clean
<NA>                  42
Industrial            11
Natural_Vegetation     4
Unknown                3
Name: count, dtype: Int64

Binary validation:
actual_binary
<NA>              45
Industrial        11
Non_Industrial     4
Name: count, dtype: Int64


In [61]:
# ============================================================
# CELL 52 — REBUILD VALIDATION SUBSET
# ============================================================

binary_df = validation_geo[
    validation_geo["actual_binary"].notna()
].copy()

print("Binary validation records:", len(binary_df))

display(
    binary_df[
        [
            "event_id",
            "actual_class_clean",
            "actual_binary",
            "confidence",
            "behavior_label"
        ]
    ].sort_values("event_id")
)

Binary validation records: 15


,event_id,actual_class_clean,actual_binary,confidence,behavior_label
0,12,Industrial,Industrial,High,Persistent
6,100,Industrial,Industrial,Medium,Persistent
9,172,Industrial,Industrial,high,Persistent
16,341,Industrial,Industrial,high,Persistent
18,479,Industrial,Industrial,Medium,Transient
19,693,Industrial,Industrial,high,Transient
26,1181,Industrial,Industrial,high,Transient
30,1225,Industrial,Industrial,high,Persistent
31,1347,Industrial,Industrial,high,Persistent
36,2074,Natural_Vegetation,Non_Industrial,medium,Transient


In [62]:
# ============================================================
# CELL 53 — INSPECT CONTEXT FEATURES
# ============================================================

osm_context_cols = [
    c for c in validation_geo.columns
    if (
        "nearest_" in c
        or "_count_" in c
        or c.startswith("wc_")
        or c == "worldcover_class"
    )
]

print("Context features:")
for c in osm_context_cols:
    print(c)

Context features:
nearest_industrial_zone_km
industrial_zone_count_375m
industrial_zone_count_1km
industrial_zone_count_3km
nearest_factory_km
factory_count_375m
factory_count_1km
factory_count_3km
nearest_mine_km
mine_count_375m
mine_count_1km
mine_count_3km
nearest_brick_km
brick_count_375m
brick_count_1km
brick_count_3km
nearest_works_km
works_count_375m
works_count_1km
works_count_3km
nearest_depot_km
depot_count_375m
depot_count_1km
depot_count_3km
nearest_power_km
power_count_375m
power_count_1km
power_count_3km
nearest_other_industry_km
other_industry_count_375m
other_industry_count_1km
other_industry_count_3km
worldcover_class
wc_tree_cover
wc_shrubland
wc_grassland
wc_cropland
wc_built_up
wc_bare_sparse
wc_snow_ice
wc_permanent_water
wc_wetland
wc_mangroves
wc_moss_lichen


In [63]:
# ============================================================
# CELL 54 — CONTEXT SUMMARY
# ============================================================

print("Context by actual class:")
print("=" * 60)

display(
    binary_df
    .groupby("actual_class_clean")[osm_context_cols]
    .agg(["count", "median", "mean"])
    .round(3)
)

Context by actual class:


nearest_industrial_zone_km                    \
                                        count   median     mean   
actual_class_clean                                                
Industrial                                 11  132.597  156.194   
Natural_Vegetation                          4  340.474  335.185   

                   industrial_zone_count_375m              \
                                        count median mean   
actual_class_clean                                          
Industrial                                 11    0.0  0.0   
Natural_Vegetation                          4    0.0  0.0   

                   industrial_zone_count_1km              \
                                       count median mean   
actual_class_clean                                         
Industrial                                11    0.0  0.0   
Natural_Vegetation                         4    0.0  0.0   

                   industrial_zone_count_3km             nearest_factory_km  \
                                       count median mean              count   
actual_class_clean                                                            
Industrial                                11    0.0  0.0                 11   
Natural_Vegetation                         4    0.0  0.0                  4   

                                     factory_count_375m              \
                     median     mean              count median mean   
actual_class_clean                                                    
Industrial          294.395  319.342                 11    0.0  0.0   
Natural_Vegetation  438.088  431.707                  4    0.0  0.0   

                   factory_count_1km             factory_count_3km         \
                               count median mean             count median   
actual_class_clean                                                          
Industrial                        11    0.0  0.0                11    0.0   
Natural_Vegetation                 4    0.0  0.0                 4    0.0   

                        nearest_mine_km                     mine_count_375m  \
                   mean           count    median      mean           count   
actual_class_clean                                                            
Industrial          0.0              11   962.144  1100.101              11   
Natural_Vegetation  0.0               4  1517.660  1530.818               4   

                               mine_count_1km             mine_count_3km  \
                   median mean          count median mean          count   
actual_class_clean                                                         
Industrial            0.0  0.0             11    0.0  0.0             11   
Natural_Vegetation    0.0  0.0              4    0.0  0.0              4   

                               nearest_brick_km                    \
                   median mean            count   median     mean   
actual_class_clean                                                  
Industrial            0.0  0.0               11  628.987  598.104   
Natural_Vegetation    0.0  0.0                4  290.660  282.475   

                   brick_count_375m             brick_count_1km              \
                              count median mean           count median mean   
actual_class_clean                                                            
Industrial                       11    0.0  0.0              11    0.0  0.0   
Natural_Vegetation                4    0.0  0.0               4    0.0  0.0   

                   brick_count_3km             nearest_works_km          \
                             count median mean            count  median   
actual_class_clean                                                        
Industrial                      11    0.0  0.0               11  59.808   
Natural_Vegetation               4    0.0  0.0                4  69.844   

                           works_count_375m

In [64]:
# ============================================================
# CELL 55 — BEHAVIOR VALIDATION
# ============================================================

print("Behavior vs actual class:")
print("=" * 60)

display(
    pd.crosstab(
        binary_df["actual_class_clean"],
        binary_df["behavior_label"]
    )
)

print("\nNormalized:")

display(
    pd.crosstab(
        binary_df["actual_class_clean"],
        binary_df["behavior_label"],
        normalize="index"
    ).mul(100).round(2)
)

Behavior vs actual class:


behavior_label,Persistent,Transient
actual_class_clean,,
Industrial,7,4
Natural_Vegetation,0,4



Normalized:


behavior_label,Persistent,Transient
actual_class_clean,,
Industrial,63.64,36.36
Natural_Vegetation,0.00,100.00


In [65]:
# ============================================================
# CELL 56 — INDUSTRIAL EVENT CONTEXT
# ============================================================

industrial_events = validation_geo[
    validation_geo["actual_class_clean"] == "Industrial"
].copy()

display(
    industrial_events[
        [
            "event_id",
            "behavior_label",
            "confidence",
            "active_days",
            "duration_days",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km"
        ] +
        [
            c for c in osm_context_cols
            if c in industrial_events.columns
        ]
    ].sort_values("event_id")
)

,event_id,behavior_label,confidence,active_days,duration_days,mean_frp,max_frp,spatial_diameter_km,nearest_industrial_zone_km,industrial_zone_count_375m,industrial_zone_count_1km,industrial_zone_count_3km,nearest_factory_km,factory_count_375m,factory_count_1km,factory_count_3km,nearest_mine_km,mine_count_375m,mine_count_1km,mine_count_3km,nearest_brick_km,brick_count_375m,brick_count_1km,brick_count_3km,nearest_works_km,works_count_375m,works_count_1km,works_count_3km,nearest_depot_km,depot_count_375m,depot_count_1km,depot_count_3km,nearest_power_km,power_count_375m,power_count_1km,power_count_3km,nearest_other_industry_km,other_industry_count_375m,other_industry_count_1km,other_industry_count_3km,worldcover_class,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_snow_ice,wc_permanent_water,wc_wetland,wc_mangroves,wc_moss_lichen
0,12,Persistent,High,17,17,0.973846,1.51,0.539481,132.597362,0,0,0,132.780696,0,0,0,619.174361,0,0,0,662.981118,0,0,0,128.310010,0,0,0,475.675349,0,0,0,NaN,0,0,0,273.485838,0,0,0,60.0,0,0,0,0,0,1,0,0,0,0,0
6,100,Persistent,Medium,26,31,1.862083,4.80,1.409461,44.015799,0,0,0,319.303903,0,0,0,407.627108,0,0,0,888.746632,0,0,0,7.647401,0,0,0,599.964016,0,0,0,NaN,0,0,0,120.990298,0,0,0,60.0,0,0,0,0,0,1,0,0,0,0,0
9,172,Persistent,high,16,16,1.443429,2.83,1.185706,129.262480,0,0,0,234.167092,0,0,0,498.076911,0,0,0,784.752536,0,0,0,12.632172,0,0,0,594.372363,0,0,0,NaN,0,0,0,188.791237,0,0,0,10.0,1,0,0,0,0,0,0,0,0,0,0
16,341,Persistent,high,20,24,1.697059,3.52,0.887156,247.910511,0,0,0,477.836281,0,0,0,1535.161130,0,0,0,628.986765,0,0,0,87.807182,0,0,0,495.198331,0,0,0,NaN,0,0,0,421.261048,0,0,0,50.0,0,0,0,0,1,0,0,0,0,0,0
18,479,Transient,Medium,2,2,0.790000,0.96,0.259333,132.188123,0,0,0,272.836367,0,0,0,1270.549571,0,0,0,301.233213,0,0,0,16.428585,0,0,0,132.188123,0,0,0,NaN,0,0,0,258.272316,0,0,0,60.0,0,0,0,0,0,1,0,0,0,0,0
19,693,Transient,high,1,1,0.660000,0.66,0.000000,50.006706,0,0,0,404.604189,0,0,0,2017.129658,0,0,0,529.438181,0,0,0,24.223901,0,0,0,260.604152,0,0,0,NaN,0,0,0,50.006706,0,0,0,10.0,1,0,0,0,0,0,0,0,0,0,0
26,1181,Transient,high,1,1,1.040000,1.04,0.000000,191.334864,0,0,0,340.754117,0,0,0,1787.911040,0,0,0,309.779307,0,0,0,66.524461,0,0,0,229.467075,0,0,0,NaN,0,0,0,230.222527,0,0,0,50.0,0,0,0,0,1,0,0,0,0,0,0
30,1225,Persistent,high,20,23,1.059375,2.31,1.106945,127.948861,0,0,0,294.394900,0,0,0,751.992821,0,0,0,794.975011,0,0,0,52.145422,0,0,0,683.506881,0,0,0,NaN,0,0,0,127.948861,0,0,0,60.0,0,0,0,0,0,1,0,0,0,0,0
31,1347,Persistent,high,20,22,1.218333,1.91,0.688072,302.961285,0,0,0,626.460853,0,0,0,1542.120700,0,0,0,634.628558,0,0,0,88.085536,0,0,0,410.790249,0,0,0,NaN,0,0,0,302.961285,0,0,0,50.0,0,0,0,0,1,0,0,0,0,0,0
47,3393,Persistent,high,7,10,2.091724,4.99,2.019369,157.676951,0,0,0,157.768035,0,0,0,709.227297,0,0,0,566.416215,0,0,0,59.807534,0,0,0,370.232383,0,0,0,NaN,0,0,0,366.066901,0,0,0,60.0,0,0,0,0,0,1,0,0,0,0,0


In [66]:
# ============================================================
# CELL 57 — NATURAL VEGETATION EVENT CONTEXT
# ============================================================

natural_events = validation_geo[
    validation_geo["actual_class_clean"] == "Natural_Vegetation"
].copy()

display(
    natural_events[
        [
            "event_id",
            "behavior_label",
            "confidence",
            "active_days",
            "duration_days",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km"
        ] +
        [
            c for c in osm_context_cols
            if c in natural_events.columns
        ]
    ].sort_values("event_id")
)

,event_id,behavior_label,confidence,active_days,duration_days,mean_frp,max_frp,spatial_diameter_km,nearest_industrial_zone_km,industrial_zone_count_375m,industrial_zone_count_1km,industrial_zone_count_3km,nearest_factory_km,factory_count_375m,factory_count_1km,factory_count_3km,nearest_mine_km,mine_count_375m,mine_count_1km,mine_count_3km,nearest_brick_km,brick_count_375m,brick_count_1km,brick_count_3km,nearest_works_km,works_count_375m,works_count_1km,works_count_3km,nearest_depot_km,depot_count_375m,depot_count_1km,depot_count_3km,nearest_power_km,power_count_375m,power_count_1km,power_count_3km,nearest_other_industry_km,other_industry_count_375m,other_industry_count_1km,other_industry_count_3km,worldcover_class,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_snow_ice,wc_permanent_water,wc_wetland,wc_mangroves,wc_moss_lichen
36,2074,Transient,medium,1,1,2.13,2.13,0.0,205.112865,0,0,0,306.791188,0,0,0,1343.178887,0,0,0,202.143128,0,0,0,79.464671,0,0,0,304.490844,0,0,0,NaN,0,0,0,359.718661,0,0,0,30.0,0,0,1,0,0,0,0,0,0,0,0
37,2201,Transient,medium,1,1,2.36,2.36,0.0,475.835336,0,0,0,569.385115,0,0,0,1683.270874,0,0,0,379.177750,0,0,0,60.223701,0,0,0,560.714516,0,0,0,NaN,0,0,0,614.551357,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0
43,2882,Transient,high,1,1,1.72,1.72,0.0,149.220109,0,0,0,252.452289,0,0,0,1352.049612,0,0,0,143.603404,0,0,0,26.447652,0,0,0,248.958387,0,0,0,NaN,0,0,0,305.076889,0,0,0,10.0,1,0,0,0,0,0,0,0,0,0,0
48,3416,Transient,high,1,1,1.72,1.72,0.0,510.571397,0,0,0,598.198421,0,0,0,1744.774502,0,0,0,404.973739,0,0,0,99.882698,0,0,0,588.523595,0,0,0,NaN,0,0,0,640.086556,0,0,0,NaN,0,0,0,0,0,0,0,0,0,0,0


In [67]:
# ============================================================
# CELL 58 — INDUSTRIAL vs NATURAL
# ============================================================

comparison_df = validation_geo[
    validation_geo["actual_class_clean"].isin([
        "Industrial",
        "Natural_Vegetation"
    ])
].copy()

comparison_features = [
    "mean_frp",
    "max_frp",
    "active_days",
    "duration_days",
    "detection_count",
    "spatial_diameter_km"
] + osm_context_cols

comparison_features = [
    c for c in comparison_features
    if c in comparison_df.columns
]

display(
    comparison_df
    .groupby("actual_class_clean")[comparison_features]
    .mean()
    .round(3)
)

,mean_frp,max_frp,active_days,duration_days,detection_count,spatial_diameter_km,nearest_industrial_zone_km,industrial_zone_count_375m,industrial_zone_count_1km,industrial_zone_count_3km,nearest_factory_km,factory_count_375m,factory_count_1km,factory_count_3km,nearest_mine_km,mine_count_375m,mine_count_1km,mine_count_3km,nearest_brick_km,brick_count_375m,brick_count_1km,brick_count_3km,nearest_works_km,works_count_375m,works_count_1km,works_count_3km,nearest_depot_km,depot_count_375m,depot_count_1km,depot_count_3km,nearest_power_km,power_count_375m,power_count_1km,power_count_3km,nearest_other_industry_km,other_industry_count_375m,other_industry_count_1km,other_industry_count_3km,worldcover_class,wc_tree_cover,wc_shrubland,wc_grassland,wc_cropland,wc_built_up,wc_bare_sparse,wc_snow_ice,wc_permanent_water,wc_wetland,wc_mangroves,wc_moss_lichen
actual_class_clean,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Industrial,1.228,2.291,11.909,13.455,28.727,0.736,156.194,0.0,0.0,0.0,319.342,0.0,0.0,0.0,1100.101,0.0,0.0,0.0,598.104,0.0,0.0,0.0,57.209,0.0,0.0,0.0,423.687,0.0,0.0,0.0,NaN,0.0,0.0,0.0,249.770,0.0,0.0,0.0,48.182,0.182,0.0,0.00,0.0,0.273,0.545,0.0,0.0,0.0,0.0,0.0
Natural_Vegetation,1.982,1.982,1.000,1.000,1.000,0.000,335.185,0.0,0.0,0.0,431.707,0.0,0.0,0.0,1530.818,0.0,0.0,0.0,282.475,0.0,0.0,0.0,66.505,0.0,0.0,0.0,425.672,0.0,0.0,0.0,NaN,0.0,0.0,0.0,479.858,0.0,0.0,0.0,20.000,0.250,0.0,0.25,0.0,0.000,0.000,0.0,0.0,0.0,0.0,0.0


In [68]:
# ============================================================
# CELL 59 — EVIDENCE AVAILABILITY
# ============================================================

print("Evidence availability for labeled events")
print("=" * 60)

for col in osm_context_cols:

    if col not in binary_df.columns:
        continue

    if "_count_" in col:

        available = (
            binary_df[col] > 0
        ).sum()

    elif "nearest_" in col:

        available = (
            binary_df[col].notna()
        ).sum()

    elif col.startswith("wc_"):

        available = (
            binary_df[col] == 1
        ).sum()

    else:
        available = binary_df[col].notna().sum()

    print(
        f"{col:50s}: {available}/{len(binary_df)}"
    )

Evidence availability for labeled events
nearest_industrial_zone_km                        : 15/15
industrial_zone_count_375m                        : 0/15
industrial_zone_count_1km                         : 0/15
industrial_zone_count_3km                         : 0/15
nearest_factory_km                                : 15/15
factory_count_375m                                : 0/15
factory_count_1km                                 : 0/15
factory_count_3km                                 : 0/15
nearest_mine_km                                   : 15/15
mine_count_375m                                   : 0/15
mine_count_1km                                    : 0/15
mine_count_3km                                    : 0/15
nearest_brick_km                                  : 15/15
brick_count_375m                                  : 0/15
brick_count_1km                                   : 0/15
brick_count_3km                                   : 0/15
nearest_works_km                           

In [69]:
# ============================================================
# CELL 60 — NORMALIZE CONFIDENCE
# ============================================================

binary_df["confidence_clean"] = (
    binary_df["confidence"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(
    binary_df["confidence_clean"]
    .value_counts()
)

confidence_clean
high      10
medium     5
Name: count, dtype: Int64


In [70]:
# ============================================================
# CELL 61 — BEHAVIORAL VALIDATION METRICS
# ============================================================

binary_df["behavior_prediction"] = np.where(
    binary_df["behavior_label"] == "Persistent",
    "Industrial",
    "Non_Industrial"
)

print("Behavior-based validation:")
print("=" * 50)

display(
    pd.crosstab(
        binary_df["actual_binary"],
        binary_df["behavior_prediction"],
        margins=True
    )
)

correct = (
    binary_df["actual_binary"]
    == binary_df["behavior_prediction"]
).sum()

accuracy = correct / len(binary_df)

print(
    f"\nAccuracy: {accuracy:.3f} "
    f"({accuracy*100:.2f}%)"
)

Behavior-based validation:


behavior_prediction,Industrial,Non_Industrial,All
actual_binary,,,
Industrial,7,4,11
Non_Industrial,0,4,4
All,7,8,15



Accuracy: 0.733 (73.33%)


In [71]:
# ============================================================
# CELL 62 — INDUSTRIAL RECALL + NATURAL REJECTION
# ============================================================

industrial = (
    binary_df["actual_binary"] == "Industrial"
)

natural = (
    binary_df["actual_binary"] == "Non_Industrial"
)

industrial_detected = (
    industrial &
    (binary_df["behavior_prediction"] == "Industrial")
).sum()

natural_rejected = (
    natural &
    (binary_df["behavior_prediction"] == "Non_Industrial")
).sum()

industrial_recall = (
    industrial_detected / industrial.sum()
)

natural_rejection = (
    natural_rejected / natural.sum()
)

print(
    f"Industrial recall: "
    f"{industrial_recall:.3f} "
    f"({industrial_recall*100:.2f}%)"
)

print(
    f"Non-industrial rejection: "
    f"{natural_rejection:.3f} "
    f"({natural_rejection*100:.2f}%)"
)

Industrial recall: 0.636 (63.64%)
Non-industrial rejection: 1.000 (100.00%)


In [72]:
# ============================================================
# CELL 63 — INDUSTRIAL FALSE NEGATIVES
# ============================================================

industrial_transient = binary_df[
    (binary_df["actual_binary"] == "Industrial") &
    (binary_df["behavior_label"] == "Transient")
].copy()

print(
    "Industrial events classified as Transient:",
    len(industrial_transient)
)

display(
    industrial_transient[
        [
            "event_id",
            "confidence",
            "active_days",
            "duration_days",
            "detection_count",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km",
            "nearest_industrial_zone_km",
            "nearest_factory_km",
            "nearest_works_km"
        ]
    ].sort_values("event_id")
)

Industrial events classified as Transient: 4


,event_id,confidence,active_days,duration_days,detection_count,mean_frp,max_frp,spatial_diameter_km,nearest_industrial_zone_km,nearest_factory_km,nearest_works_km
18,479,Medium,2,2,2,0.79,0.96,0.259333,132.188123,272.836367,16.428585
19,693,high,1,1,1,0.66,0.66,0.000000,50.006706,404.604189,24.223901
26,1181,high,1,1,1,1.04,1.04,0.000000,191.334864,340.754117,66.524461
55,3906,medium,1,1,1,0.67,0.67,0.000000,202.228141,251.857858,85.685083


In [73]:
# ============================================================
# CELL 64 — NON-INDUSTRIAL FALSE POSITIVES
# ============================================================

nonindustrial_persistent = binary_df[
    (binary_df["actual_binary"] == "Non_Industrial") &
    (binary_df["behavior_label"] == "Persistent")
].copy()

print(
    "Non-industrial events classified as Persistent:",
    len(nonindustrial_persistent)
)

display(
    nonindustrial_persistent[
        [
            "event_id",
            "confidence",
            "active_days",
            "duration_days",
            "detection_count",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km",
            "nearest_industrial_zone_km",
            "nearest_factory_km",
            "nearest_works_km"
        ]
    ].sort_values("event_id")
)

Non-industrial events classified as Persistent: 0


,event_id,confidence,active_days,duration_days,detection_count,mean_frp,max_frp,spatial_diameter_km,nearest_industrial_zone_km,nearest_factory_km,nearest_works_km


In [74]:
# ============================================================
# CELL 65 — CONTEXT EVIDENCE FLAGS
# ============================================================

binary_df["osm_industrial_nearby"] = (
    (
        binary_df["nearest_industrial_zone_km"] <= 3
    ) |
    (
        binary_df["nearest_factory_km"] <= 3
    ) |
    (
        binary_df["nearest_works_km"] <= 3
    ) |
    (
        binary_df["nearest_mine_km"] <= 3
    ) |
    (
        binary_df["nearest_brick_km"] <= 3
    )
)

binary_df["built_up_context"] = (
    binary_df["wc_built_up"] == 1
)

binary_df["vegetation_context"] = (
    (
        binary_df["wc_tree_cover"] == 1
    ) |
    (
        binary_df["wc_grassland"] == 1
    ) |
    (
        binary_df["wc_cropland"] == 1
    )
)

display(
    binary_df[
        [
            "event_id",
            "actual_class_clean",
            "behavior_label",
            "osm_industrial_nearby",
            "built_up_context",
            "vegetation_context"
        ]
    ]
)

,event_id,actual_class_clean,behavior_label,osm_industrial_nearby,built_up_context,vegetation_context
0,12,Industrial,Persistent,False,False,False
6,100,Industrial,Persistent,False,False,False
9,172,Industrial,Persistent,False,False,True
16,341,Industrial,Persistent,False,True,False
18,479,Industrial,Transient,False,False,False
19,693,Industrial,Transient,False,False,True
26,1181,Industrial,Transient,False,True,False
30,1225,Industrial,Persistent,False,False,False
31,1347,Industrial,Persistent,False,True,False
36,2074,Natural_Vegetation,Transient,False,False,True


In [75]:
# ============================================================
# CELL 66 — FINAL EVIDENCE MATRIX
# ============================================================

final_evidence = binary_df[
    [
        "event_id",
        "actual_class_clean",
        "confidence_clean",

        "behavior_label",

        "mean_frp",
        "max_frp",
        "active_days",
        "duration_days",
        "detection_count",
        "spatial_diameter_km",

        "nearest_industrial_zone_km",
        "nearest_factory_km",
        "nearest_works_km",
        "nearest_mine_km",
        "nearest_brick_km",

        "osm_industrial_nearby",
        "built_up_context",
        "vegetation_context"
    ]
].sort_values("event_id")

display(final_evidence)

,event_id,actual_class_clean,confidence_clean,behavior_label,mean_frp,max_frp,active_days,duration_days,detection_count,spatial_diameter_km,nearest_industrial_zone_km,nearest_factory_km,nearest_works_km,nearest_mine_km,nearest_brick_km,osm_industrial_nearby,built_up_context,vegetation_context
0,12,Industrial,high,Persistent,0.973846,1.51,17,17,26,0.539481,132.597362,132.780696,128.310010,619.174361,662.981118,False,False,False
6,100,Industrial,medium,Persistent,1.862083,4.80,26,31,96,1.409461,44.015799,319.303903,7.647401,407.627108,888.746632,False,False,False
9,172,Industrial,high,Persistent,1.443429,2.83,16,16,35,1.185706,129.262480,234.167092,12.632172,498.076911,784.752536,False,False,True
16,341,Industrial,high,Persistent,1.697059,3.52,20,24,34,0.887156,247.910511,477.836281,87.807182,1535.161130,628.986765,False,True,False
18,479,Industrial,medium,Transient,0.790000,0.96,2,2,2,0.259333,132.188123,272.836367,16.428585,1270.549571,301.233213,False,False,False
19,693,Industrial,high,Transient,0.660000,0.66,1,1,1,0.000000,50.006706,404.604189,24.223901,2017.129658,529.438181,False,False,True
26,1181,Industrial,high,Transient,1.040000,1.04,1,1,1,0.000000,191.334864,340.754117,66.524461,1787.911040,309.779307,False,True,False
30,1225,Industrial,high,Persistent,1.059375,2.31,20,23,32,1.106945,127.948861,294.394900,52.145422,751.992821,794.975011,False,False,False
31,1347,Industrial,high,Persistent,1.218333,1.91,20,22,30,0.688072,302.961285,626.460853,88.085536,1542.120700,634.628558,False,True,False
36,2074,Natural_Vegetation,medium,Transient,2.130000,2.13,1,1,1,0.000000,205.112865,306.791188,79.464671,1343.178887,202.143128,False,False,True


In [76]:
# ============================================================
# CELL 67 — SAVE FINAL VALIDATION EVIDENCE
# ============================================================

FINAL_VALIDATION_FILE = (
    "viirs_validation_evidence_v2.csv"
)

final_evidence.to_csv(
    FINAL_VALIDATION_FILE,
    index=False
)

print(
    "Saved:",
    FINAL_VALIDATION_FILE
)

print(
    "Rows:",
    len(final_evidence)
)

Saved: viirs_validation_evidence_v2.csv
Rows: 15


In [77]:
# ============================================================
# CELL 68 — OSM COVERAGE ACROSS FULL EVENT DATASET
# ============================================================

full_events = events_geo.copy()

osm_count_cols_full = [
    c for c in full_events.columns
    if "_count_" in c
]

print("OSM coverage across ALL V11 events")
print("=" * 60)

for radius in ["375m", "1km", "3km"]:

    print(f"\n--- {radius} ---")

    cols = [
        c for c in osm_count_cols_full
        if c.endswith(radius)
    ]

    for col in cols:

        n = (
            full_events[col] > 0
        ).sum()

        print(
            f"{col:50s} "
            f"{n:5d} / {len(full_events)} "
            f"({n / len(full_events) * 100:.2f}%)"
        )

OSM coverage across ALL V11 events

--- 375m ---
industrial_zone_count_375m                             0 / 4893 (0.00%)
factory_count_375m                                     0 / 4893 (0.00%)
mine_count_375m                                        0 / 4893 (0.00%)
brick_count_375m                                       0 / 4893 (0.00%)
works_count_375m                                       6 / 4893 (0.12%)
depot_count_375m                                       0 / 4893 (0.00%)
power_count_375m                                       0 / 4893 (0.00%)
other_industry_count_375m                              0 / 4893 (0.00%)

--- 1km ---
industrial_zone_count_1km                              0 / 4893 (0.00%)
factory_count_1km                                      0 / 4893 (0.00%)
mine_count_1km                                         0 / 4893 (0.00%)
brick_count_1km                                        0 / 4893 (0.00%)
works_count_1km                                       21 / 4893 (0.43%)
de

In [79]:
# ============================================================
# CELL 69 — OSM DISTANCE DISTRIBUTION
# ============================================================

nearest_full_cols = [
    c for c in full_events.columns
    if c.startswith("nearest_")
]

distance_summary = (
    full_events[nearest_full_cols]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95]
    )
    .T
)

display(
    distance_summary[
        [
            "count",
            "mean",
            "50%",
            "75%",
            "90%",
            "95%",
            "min"
        ]
    ].round(3)
)

,count,mean,50%,75%,90%,95%,min
nearest_industrial_zone_km,4893.0,172.054,155.892,207.335,312.617,451.428,2.077
nearest_factory_km,4893.0,370.546,303.133,442.677,670.374,885.658,6.740
nearest_mine_km,4893.0,1322.940,1375.056,1775.929,1896.743,1977.478,115.464
nearest_brick_km,4893.0,494.418,423.196,627.852,882.155,967.035,5.598
nearest_works_km,4893.0,56.244,49.271,76.281,105.743,124.097,0.130
nearest_depot_km,4893.0,338.393,296.823,478.372,616.006,654.290,2.216
nearest_power_km,0.0,NaN,NaN,NaN,NaN,NaN,NaN
nearest_other_industry_km,4893.0,283.650,261.606,356.293,575.699,697.346,2.077


In [80]:
# ============================================================
# CELL 70 — OSM EVIDENCE AT MULTIPLE DISTANCES
# ============================================================

distance_thresholds = [0.375, 1, 3, 5, 10, 25, 50]

entity_nearest_cols = [
    c for c in nearest_full_cols
    if c != "nearest_any_industry_km"
]

print("Events with at least one OSM industrial-context entity")
print("=" * 65)

for threshold in distance_thresholds:

    nearby = (
        full_events[entity_nearest_cols]
        <= threshold
    ).any(axis=1)

    print(
        f"{threshold:6.3f} km : "
        f"{nearby.sum():5d} / {len(full_events)} "
        f"({nearby.mean()*100:.2f}%)"
    )

Events with at least one OSM industrial-context entity
 0.375 km :     6 / 4893 (0.12%)
 1.000 km :    21 / 4893 (0.43%)
 3.000 km :   100 / 4893 (2.04%)
 5.000 km :   165 / 4893 (3.37%)
10.000 km :   437 / 4893 (8.93%)
25.000 km :  1185 / 4893 (24.22%)
50.000 km :  2527 / 4893 (51.65%)


In [81]:
# ============================================================
# CELL 71 — BEHAVIOR × OSM COVERAGE
# ============================================================

behavior_full = validation_analysis[
    [
        "event_id",
        "behavior_label"
    ]
].drop_duplicates("event_id")

behavior_full = behavior_full.merge(
    full_events[
        ["event_id"] + nearest_full_cols
    ],
    on="event_id",
    how="left"
)

for threshold in [1, 3, 5, 10]:

    behavior_full[
        f"osm_near_{threshold}km"
    ] = (
        behavior_full[entity_nearest_cols]
        <= threshold
    ).any(axis=1)

print("OSM contextual presence by behavioral group:")

display(
    behavior_full
    .groupby("behavior_label")[
        [
            "osm_near_1km",
            "osm_near_3km",
            "osm_near_5km",
            "osm_near_10km"
        ]
    ]
    .mean()
    .mul(100)
    .round(2)
)

OSM contextual presence by behavioral group:


,osm_near_1km,osm_near_3km,osm_near_5km,osm_near_10km
behavior_label,,,,
Persistent,0.0,0.0,0.0,12.5
Transient,0.0,10.0,10.0,10.0


In [82]:
# ============================================================
# CELL 72 — POTENTIAL CONTEXTUAL RESCUE EVENTS
# ============================================================

rescue_candidates = behavior_full[
    (
        behavior_full["behavior_label"]
        == "Transient"
    )
    &
    (
        behavior_full["osm_near_3km"]
    )
].copy()

print(
    "Transient events with OSM context within 3 km:",
    len(rescue_candidates)
)

display(
    rescue_candidates.sort_values("event_id").head(50)
)

Transient events with OSM context within 3 km: 1


,event_id,behavior_label,nearest_industrial_zone_km,nearest_factory_km,nearest_mine_km,nearest_brick_km,nearest_works_km,nearest_depot_km,nearest_power_km,nearest_other_industry_km,osm_near_1km,osm_near_3km,osm_near_5km,osm_near_10km
5,3859,Transient,3.812268,147.738198,2109.374131,623.532711,2.853806,3.812268,NaN,57.336886,False,True,True,True


In [83]:
# ============================================================
# CELL 73 — RESCUE CANDIDATE THERMAL PROFILE
# ============================================================

rescue_candidates = rescue_candidates.merge(
    full_events[
        [
            "event_id",
            "mean_frp",
            "max_frp",
            "active_days",
            "duration_days",
            "detection_count",
            "spatial_diameter_km"
        ]
    ],
    on="event_id",
    how="left"
)

display(
    rescue_candidates[
        [
            "event_id",
            "behavior_label",
            "mean_frp",
            "max_frp",
            "active_days",
            "duration_days",
            "detection_count",
            "spatial_diameter_km",
            "osm_near_1km",
            "osm_near_3km",
            "osm_near_5km",
            "osm_near_10km"
        ]
    ]
    .sort_values(
        ["active_days", "max_frp"],
        ascending=False
    )
    .head(50)
)

,event_id,behavior_label,mean_frp,max_frp,active_days,duration_days,detection_count,spatial_diameter_km,osm_near_1km,osm_near_3km,osm_near_5km,osm_near_10km
0,3859,Transient,1.556,2.39,2,2,5,0.585093,False,True,True,True


In [84]:
# ============================================================
# CELL 74 — SAVE CONTEXTUAL RESCUE CANDIDATES
# ============================================================

rescue_candidates.to_csv(
    "viirs_context_rescue_candidates_v1.csv",
    index=False
)

print(
    "Saved viirs_context_rescue_candidates_v1.csv"
)

Saved viirs_context_rescue_candidates_v1.csv


In [87]:
# ============================================================
# CELL 75 — PREPARE FULL VALIDATION DATASET
# ============================================================

# validation_geo already contains:
# - actual labels
# - behavior labels
# - OSM context
# - WorldCover context

df = validation_geo.copy()

df["actual_class_clean"] = (
    df["actual_class"]
    .astype("string")
    .str.strip()
)

df["actual_binary"] = (
    df["actual_class_clean"]
    .map({
        "Industrial": "Industrial",
        "Agricultural": "Non_Industrial",
        "Natural_Vegetation": "Non_Industrial",
        "Other": "Non_Industrial"
    })
)

# Only classes usable for binary validation
df = df[
    df["actual_binary"].notna()
].copy()

print("Usable labeled events:", len(df))

print("\nActual class distribution:")
display(
    df["actual_class_clean"].value_counts()
)

print("\nBinary distribution:")
display(
    df["actual_binary"].value_counts()
)

Usable labeled events: 15

Actual class distribution:


actual_class_clean
Industrial            11
Natural_Vegetation     4
Name: count, dtype: Int64


Binary distribution:


actual_binary
Industrial        11
Non_Industrial     4
Name: count, dtype: int64

In [88]:
# ============================================================
# CELL 76 — DOMAIN EVIDENCE SCORE
# ============================================================

# ------------------------------------------------------------
# 1. BEHAVIOR
# ------------------------------------------------------------

df["score_persistent"] = (
    (df["behavior_label"] == "Persistent").astype(int) * 2
)

df["score_high_activity"] = (
    (df["active_days"] >= 5).astype(int)
)

df["score_high_frp"] = (
    (df["max_frp"] >= df["max_frp"].median()).astype(int)
)

# ------------------------------------------------------------
# 2. OSM INDUSTRIAL CONTEXT
# ------------------------------------------------------------

osm_dist_cols = [
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_works_km",
    "nearest_mine_km",
    "nearest_brick_km"
]

osm_dist_cols = [
    c for c in osm_dist_cols
    if c in df.columns
]

df["osm_industrial_3km"] = (
    df[osm_dist_cols]
    .le(3)
    .any(axis=1)
)

df["osm_industrial_10km"] = (
    df[osm_dist_cols]
    .le(10)
    .any(axis=1)
)

# Strong evidence if explicitly nearby
df["score_osm_3km"] = (
    df["osm_industrial_3km"].astype(int) * 4
)

# Weak evidence at larger distance
df["score_osm_10km"] = (
    (
        df["osm_industrial_10km"] &
        ~df["osm_industrial_3km"]
    ).astype(int)
)

# ------------------------------------------------------------
# 3. WORLDCOVER
# ------------------------------------------------------------

df["score_builtup"] = (
    (df["wc_built_up"] == 1).astype(int) * 2
)

df["score_vegetation"] = (
    (
        (df["wc_tree_cover"] == 1) |
        (df["wc_grassland"] == 1) |
        (df["wc_cropland"] == 1)
    ).astype(int) * -2
)

# ------------------------------------------------------------
# 4. FINAL DOMAIN SCORE
# ------------------------------------------------------------

score_cols = [
    "score_persistent",
    "score_high_activity",
    "score_high_frp",
    "score_osm_3km",
    "score_osm_10km",
    "score_builtup",
    "score_vegetation"
]

df["domain_score"] = df[score_cols].sum(axis=1)

display(
    df[
        [
            "event_id",
            "actual_class_clean",
            "behavior_label",
            "domain_score",
            "osm_industrial_3km",
            "osm_industrial_10km",
            "wc_built_up",
            "wc_tree_cover",
            "wc_grassland",
            "wc_cropland"
        ]
    ]
    .sort_values("domain_score", ascending=False)
)

,event_id,actual_class_clean,behavior_label,domain_score,osm_industrial_3km,osm_industrial_10km,wc_built_up,wc_tree_cover,wc_grassland,wc_cropland
16,341,Industrial,Persistent,6,False,False,1,0,0,0
31,1347,Industrial,Persistent,6,False,False,1,0,0,0
6,100,Industrial,Persistent,5,False,True,0,0,0,0
30,1225,Industrial,Persistent,4,False,False,0,0,0,0
47,3393,Industrial,Persistent,4,False,False,0,0,0,0
0,12,Industrial,Persistent,3,False,False,0,0,0,0
9,172,Industrial,Persistent,2,False,False,0,1,0,0
26,1181,Industrial,Transient,2,False,False,1,0,0,0
37,2201,Natural_Vegetation,Transient,1,False,False,0,0,0,0
55,3906,Industrial,Transient,0,False,False,0,0,0,0


In [89]:
# ============================================================
# CELL 77 — FINAL DOMAIN CLASSIFICATION
# ============================================================

df["final_prediction"] = np.select(
    [
        df["domain_score"] >= 4,
        df["domain_score"] <= 0
    ],
    [
        "Industrial",
        "Non_Industrial"
    ],
    default="Uncertain"
)

print("Final prediction distribution:")
display(df["final_prediction"].value_counts())

display(
    df[
        [
            "event_id",
            "actual_class_clean",
            "behavior_label",
            "domain_score",
            "final_prediction"
        ]
    ].sort_values("event_id")
)

Final prediction distribution:


final_prediction
Non_Industrial    6
Industrial        5
Uncertain         4
Name: count, dtype: int64

,event_id,actual_class_clean,behavior_label,domain_score,final_prediction
0,12,Industrial,Persistent,3,Uncertain
6,100,Industrial,Persistent,5,Industrial
9,172,Industrial,Persistent,2,Uncertain
16,341,Industrial,Persistent,6,Industrial
18,479,Industrial,Transient,0,Non_Industrial
19,693,Industrial,Transient,-2,Non_Industrial
26,1181,Industrial,Transient,2,Uncertain
30,1225,Industrial,Persistent,4,Industrial
31,1347,Industrial,Persistent,6,Industrial
36,2074,Natural_Vegetation,Transient,-1,Non_Industrial


In [90]:
# ============================================================
# CELL 78 — FINAL VALIDATION METRICS
# ============================================================

evaluated = df[
    df["final_prediction"] != "Uncertain"
].copy()

print("Evaluated events:", len(evaluated))
print("Uncertain events:", (df["final_prediction"] == "Uncertain").sum())

print("\nConfusion Matrix:")
display(
    pd.crosstab(
        evaluated["actual_binary"],
        evaluated["final_prediction"],
        margins=True
    )
)

accuracy = (
    evaluated["actual_binary"]
    == evaluated["final_prediction"]
).mean()

industrial_mask = (
    evaluated["actual_binary"] == "Industrial"
)

industrial_recall = (
    (
        evaluated.loc[industrial_mask, "final_prediction"]
        == "Industrial"
    ).mean()
)

nonindustrial_mask = (
    evaluated["actual_binary"] == "Non_Industrial"
)

nonindustrial_rejection = (
    (
        evaluated.loc[
            nonindustrial_mask,
            "final_prediction"
        ]
        == "Non_Industrial"
    ).mean()
)

print(f"\nAccuracy: {accuracy:.3f} ({accuracy*100:.2f}%)")
print(
    f"Industrial recall: "
    f"{industrial_recall:.3f} "
    f"({industrial_recall*100:.2f}%)"
)
print(
    f"Non-industrial rejection: "
    f"{nonindustrial_rejection:.3f} "
    f"({nonindustrial_rejection*100:.2f}%)"
)

Evaluated events: 11
Uncertain events: 4

Confusion Matrix:


final_prediction,Industrial,Non_Industrial,All
actual_binary,,,
Industrial,5,3,8
Non_Industrial,0,3,3
All,5,6,11



Accuracy: 0.727 (72.73%)
Industrial recall: 0.625 (62.50%)
Non-industrial rejection: 1.000 (100.00%)


In [91]:
# ============================================================
# CELL 79 — FINAL ERROR ANALYSIS
# ============================================================

errors = evaluated[
    evaluated["actual_binary"]
    != evaluated["final_prediction"]
].copy()

print("Final classification errors:", len(errors))

display(
    errors[
        [
            "event_id",
            "actual_class_clean",
            "behavior_label",
            "domain_score",
            "final_prediction",
            "active_days",
            "duration_days",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km",
            "osm_industrial_3km",
            "osm_industrial_10km",
            "wc_built_up"
        ]
    ].sort_values("event_id")
)

Final classification errors: 3


,event_id,actual_class_clean,behavior_label,domain_score,final_prediction,active_days,duration_days,mean_frp,max_frp,spatial_diameter_km,osm_industrial_3km,osm_industrial_10km,wc_built_up
18,479,Industrial,Transient,0,Non_Industrial,2,2,0.79,0.96,0.259333,False,False,0
19,693,Industrial,Transient,-2,Non_Industrial,1,1,0.66,0.66,0.000000,False,False,0
55,3906,Industrial,Transient,0,Non_Industrial,1,1,0.67,0.67,0.000000,False,False,0


In [92]:
# ============================================================
# CELL 80 — SAVE FINAL VALIDATION OUTPUT
# ============================================================

FINAL_COLUMNS = [
    "event_id",
    "centroid_lat",
    "centroid_lon",
    "start_date",
    "end_date",
    "active_days",
    "duration_days",
    "detection_count",
    "mean_frp",
    "max_frp",
    "spatial_diameter_km",
    "behavior_label",
    "domain_score",
    "final_prediction"
]

final_validation_output = df[
    [c for c in FINAL_COLUMNS if c in df.columns]
].copy()

final_validation_output.to_csv(
    "viirs_final_validation_predictions.csv",
    index=False
)

print(
    "Saved: viirs_final_validation_predictions.csv"
)

print(
    "Rows:",
    len(final_validation_output)
)

Saved: viirs_final_validation_predictions.csv
Rows: 15


In [93]:
# ============================================================
# CELL 81 — LOAD UPDATED 60-EVENT VALIDATION
# ============================================================

VALIDATION_FILE = "VIIRS_60_Event_Validation_Review_Pack (1).xlsx"

validation_new = pd.read_excel(
    VALIDATION_FILE,
    sheet_name="Validation Review"
)

print("Validation shape:", validation_new.shape)

print("\nColumns:")
print(validation_new.columns.tolist())

print("\nActual class distribution:")
display(
    validation_new["actual_class"]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
)

Validation shape: (60, 31)

Columns:
['review_id', 'event_id', 'latitude', 'longitude', 'start_date', 'end_date', 'active_days', 'duration_days', 'detection_count', 'daily_object_count', 'spatial_diameter_km', 'mean_frp', 'max_frp', 'mean_bright_ti4', 'max_bright_ti4', 'mean_bright_ti5', 'max_bright_ti5', 'temporal_group', 'satellite_map_link', 'osm_map_link', 'actual_class', 'confidence', 'industrial_evidence', 'agricultural_evidence', 'natural_evidence', 'other_evidence', 'evidence_notes', 'imagery_checked', 'osm_checked', 'review_date', 'reviewer_notes']

Actual class distribution:


actual_class
Industrial            35
Natural_Vegetation    15
Unknown                5
Agricultural           3
Other                  2
Name: count, dtype: Int64

In [94]:
# ============================================================
# CELL 82 — VALIDATION LABEL QC
# ============================================================

validation_new["actual_class_clean"] = (
    validation_new["actual_class"]
    .astype("string")
    .str.strip()
)

print("Total validation events:", len(validation_new))
print("Unique event IDs:", validation_new["event_id"].nunique())

print("\nClasses:")
display(
    validation_new["actual_class_clean"]
    .value_counts(dropna=False)
)

print("\nMissing labels:",
      validation_new["actual_class_clean"].isna().sum())

print("\nDuplicate event IDs:",
      validation_new["event_id"].duplicated().sum())

Total validation events: 60
Unique event IDs: 60

Classes:


actual_class_clean
Industrial            35
Natural_Vegetation    15
Unknown                5
Agricultural           3
Other                  2
Name: count, dtype: Int64


Missing labels: 0

Duplicate event IDs: 0


In [95]:
# ============================================================
# CELL 83 — CREATE FINAL VALIDATION BASE
# ============================================================

validation_base = validation_new.copy()

validation_base["actual_binary"] = (
    validation_base["actual_class_clean"]
    .map({
        "Industrial": "Industrial",
        "Agricultural": "Non_Industrial",
        "Natural_Vegetation": "Non_Industrial",
        "Other": "Non_Industrial"
    })
)

print("Binary validation:")
display(
    validation_base["actual_binary"]
    .value_counts(dropna=False)
)

print("\nExcluded from binary evaluation:")
display(
    validation_base[
        validation_base["actual_binary"].isna()
    ][
        ["event_id", "actual_class_clean"]
    ]
)

Binary validation:


actual_binary
Industrial        35
Non_Industrial    20
NaN                5
Name: count, dtype: int64


Excluded from binary evaluation:


,event_id,actual_class_clean
2,1967,Unknown
4,368,Unknown
5,3859,Unknown
21,1185,Unknown
22,308,Unknown


In [96]:
# ============================================================
# CELL 84 — MERGE 60 VALIDATION EVENTS WITH MODEL FEATURES
# ============================================================

validation_full = validation_base.merge(
    validation_geo.drop(
        columns=[
            "actual_class",
            "actual_class_clean",
            "actual_binary"
        ],
        errors="ignore"
    ),
    on="event_id",
    how="left",
    suffixes=("", "_model")
)

print("Merged shape:", validation_full.shape)

print(
    "Events successfully matched:",
    validation_full["event_id"].notna().sum()
)

print(
    "Events with missing model data:",
    validation_full["behavior_label"].isna().sum()
)

print("\nValidation classes after merge:")
display(
    validation_full["actual_class_clean"]
    .value_counts(dropna=False)
)

Merged shape: (60, 111)
Events successfully matched: 60
Events with missing model data: 42

Validation classes after merge:


actual_class_clean
Industrial            35
Natural_Vegetation    15
Unknown                5
Agricultural           3
Other                  2
Name: count, dtype: Int64

In [97]:
# ============================================================
# CELL 85 — CONTEXT FEATURE QC
# ============================================================

required_context = [
    "behavior_label",
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_works_km",
    "nearest_mine_km",
    "nearest_brick_km",
    "wc_built_up",
    "wc_tree_cover",
    "wc_grassland",
    "wc_cropland"
]

context_check = pd.DataFrame({
    "feature": required_context,
    "exists": [
        c in validation_full.columns
        for c in required_context
    ]
})

display(context_check)

missing_context = context_check[
    ~context_check["exists"]
]["feature"].tolist()

if missing_context:
    raise ValueError(
        f"Missing required context features: {missing_context}"
    )

print("All required context features are available.")

,feature,exists
0,behavior_label,True
1,nearest_industrial_zone_km,True
2,nearest_factory_km,True
3,nearest_works_km,True
4,nearest_mine_km,True
5,nearest_brick_km,True
6,wc_built_up,True
7,wc_tree_cover,True
8,wc_grassland,True
9,wc_cropland,True


All required context features are available.


In [98]:
# ============================================================
# CELL 86 — INSPECT COMPLETE BEHAVIOR MODEL OUTPUT
# ============================================================

print("Full event dataset:")
print(full_events.shape)

print("\nAvailable behavior/model columns:")

behavior_candidates = [
    c for c in full_events.columns
    if (
        "cluster" in c.lower()
        or "behavior" in c.lower()
        or "isolation" in c.lower()
        or "prediction" in c.lower()
    )
]

print(behavior_candidates)

Full event dataset:
(4893, 57)

Available behavior/model columns:
[]


In [99]:
# ============================================================
# CELL 87 — FIND EXISTING BEHAVIOR MODEL OBJECTS
# ============================================================

print("Objects containing KMeans:")
print([
    name for name in globals()
    if "kmeans" in name.lower()
])

print("\nObjects containing scaler:")
print([
    name for name in globals()
    if "scaler" in name.lower()
])

Objects containing KMeans:
['KMeans', 'kmeans']

Objects containing scaler:
['RobustScaler', 'scaler']


In [100]:
# ============================================================
# CELL 88 — VERIFY EXISTING BEHAVIOR MODEL
# ============================================================

print("KMeans parameters:")
print("n_clusters:", kmeans.n_clusters)
print("random_state:", kmeans.random_state)

print("\nScaler:")
print(type(scaler).__name__)

print("\nKMeans cluster centers shape:",
      kmeans.cluster_centers_.shape)

print("\nScaler feature count:",
      len(scaler.feature_names_in_)
      if hasattr(scaler, "feature_names_in_")
      else "feature names not stored")

if hasattr(scaler, "feature_names_in_"):
    print("\nScaler features:")
    print(list(scaler.feature_names_in_))

KMeans parameters:
n_clusters: 2
random_state: 42

Scaler:
RobustScaler

KMeans cluster centers shape: (2, 11)

Scaler feature count: 11

Scaler features:
['mean_frp', 'max_frp', 'mean_bright_ti4', 'max_bright_ti4', 'mean_bright_ti5', 'max_bright_ti5', 'active_days', 'duration_days', 'activity_frequency', 'detections_per_active_day', 'spatial_diameter_km']


In [101]:
# ============================================================
# CELL 89 — BEHAVIOR PREDICTION FOR ALL V11 EVENTS
# ============================================================

behavior_features = list(scaler.feature_names_in_)

X_all_behavior = full_events[behavior_features].copy()

# Apply the EXISTING scaler
X_all_scaled = scaler.transform(X_all_behavior)

# Apply the EXISTING K-Means model
all_clusters = kmeans.predict(X_all_scaled)

full_events["behavior_cluster"] = all_clusters

print("Behavior prediction completed.")

print("\nCluster distribution:")
display(
    full_events["behavior_cluster"]
    .value_counts()
    .sort_index()
)

Behavior prediction completed.

Cluster distribution:


behavior_cluster
0    4683
1     210
Name: count, dtype: int64

In [102]:
# ============================================================
# CELL 90 — IDENTIFY BEHAVIOR CLUSTERS
# ============================================================

cluster_profile = (
    full_events
    .groupby("behavior_cluster")
    [
        [
            "active_days",
            "duration_days",
            "detection_count",
            "detections_per_active_day",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km"
        ]
    ]
    .mean()
)

display(cluster_profile)

# Persistent behavior = cluster with higher mean active_days
persistent_cluster = (
    cluster_profile["active_days"].idxmax()
)

transient_cluster = (
    cluster_profile["active_days"].idxmin()
)

full_events["behavior_label"] = np.where(
    full_events["behavior_cluster"] == persistent_cluster,
    "Persistent",
    "Transient"
)

print("\nPersistent cluster:", persistent_cluster)
print("Transient cluster:", transient_cluster)

print("\nBehavior distribution:")
display(
    full_events["behavior_label"].value_counts()
)

,active_days,duration_days,detection_count,detections_per_active_day,mean_frp,max_frp,spatial_diameter_km
behavior_cluster,,,,,,,
0,1.226778,1.333547,1.407218,1.141791,1.327267,1.415353,0.063404
1,13.114286,16.366667,33.423810,2.260882,1.434003,2.995571,0.933761



Persistent cluster: 1
Transient cluster: 0

Behavior distribution:


behavior_label
Transient     4683
Persistent     210
Name: count, dtype: int64

In [103]:
# ============================================================
# CELL 91 — MERGE NEW VALIDATION WITH COMPLETE MODEL DATA
# ============================================================

validation_full = validation_base.merge(
    full_events,
    on="event_id",
    how="left",
    suffixes=("_validation", "")
)

print("Validation rows:", len(validation_full))

print(
    "Missing behavior predictions:",
    validation_full["behavior_label"].isna().sum()
)

print(
    "Missing OSM/WorldCover rows:",
    validation_full["event_id"].isna().sum()
)

print("\nActual classes:")
display(
    validation_full["actual_class_clean"]
    .value_counts(dropna=False)
)

Validation rows: 60
Missing behavior predictions: 0
Missing OSM/WorldCover rows: 0

Actual classes:


actual_class_clean
Industrial            35
Natural_Vegetation    15
Unknown                5
Agricultural           3
Other                  2
Name: count, dtype: Int64

In [104]:
# ============================================================
# CELL 92 — FINAL MODEL INPUT SANITY CHECK
# ============================================================

required_columns = [
    "event_id",
    "actual_class_clean",
    "actual_binary",
    "behavior_cluster",
    "behavior_label",
    "active_days",
    "duration_days",
    "mean_frp",
    "max_frp",
    "spatial_diameter_km"
]

missing = [
    c for c in required_columns
    if c not in validation_full.columns
]

print("Missing required columns:", missing)

print(
    "Rows with missing behavior label:",
    validation_full["behavior_label"].isna().sum()
)

assert len(validation_full) == 60
assert validation_full["behavior_label"].notna().all()
assert len(missing) == 0

print("\nVALIDATION DATASET READY")

Missing required columns: []
Rows with missing behavior label: 0

VALIDATION DATASET READY


In [105]:
# ============================================================
# CELL 93 — BEHAVIOR CLUSTER PROFILES
# ============================================================

cluster_profile = (
    full_events
    .groupby("behavior_cluster")
    [
        [
            "active_days",
            "duration_days",
            "detection_count",
            "detections_per_active_day",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km"
        ]
    ]
    .mean()
)

display(cluster_profile)

persistent_cluster = cluster_profile["active_days"].idxmax()
transient_cluster = cluster_profile["active_days"].idxmin()

full_events["behavior_label"] = np.where(
    full_events["behavior_cluster"] == persistent_cluster,
    "Persistent",
    "Transient"
)

print("Persistent cluster:", persistent_cluster)
print("Transient cluster:", transient_cluster)

,active_days,duration_days,detection_count,detections_per_active_day,mean_frp,max_frp,spatial_diameter_km
behavior_cluster,,,,,,,
0,1.226778,1.333547,1.407218,1.141791,1.327267,1.415353,0.063404
1,13.114286,16.366667,33.423810,2.260882,1.434003,2.995571,0.933761


Persistent cluster: 1
Transient cluster: 0


In [106]:
# ============================================================
# CELL 94 — BEHAVIOR MODEL VALIDATION
# ============================================================

behavior_eval = validation_full[
    validation_full["actual_binary"].notna()
].copy()

print("Usable labeled events:", len(behavior_eval))

print("\nActual vs behavior:")
display(
    pd.crosstab(
        behavior_eval["actual_binary"],
        behavior_eval["behavior_label"],
        margins=True
    )
)

behavior_accuracy = (
    behavior_eval["actual_binary"]
    == np.where(
        behavior_eval["behavior_label"] == "Persistent",
        "Industrial",
        "Non_Industrial"
    )
).mean()

industrial_mask = (
    behavior_eval["actual_binary"] == "Industrial"
)

industrial_recall = (
    (
        behavior_eval.loc[
            industrial_mask,
            "behavior_label"
        ] == "Persistent"
    ).mean()
)

nonindustrial_mask = (
    behavior_eval["actual_binary"] == "Non_Industrial"
)

nonindustrial_rejection = (
    (
        behavior_eval.loc[
            nonindustrial_mask,
            "behavior_label"
        ] == "Transient"
    ).mean()
)

print(f"\nBehavior accuracy: {behavior_accuracy:.3f}")
print(f"Industrial recall: {industrial_recall:.3f}")
print(f"Non-industrial rejection: {nonindustrial_rejection:.3f}")

Usable labeled events: 55

Actual vs behavior:


behavior_label,Persistent,Transient,All
actual_binary,,,
Industrial,21,14,35
Non_Industrial,1,19,20
All,22,33,55



Behavior accuracy: 0.727
Industrial recall: 0.600
Non-industrial rejection: 0.950


In [107]:
# ============================================================
# CELL 95 — INDUSTRIAL TRANSIENT EVENTS
# ============================================================

industrial_transient = behavior_eval[
    (behavior_eval["actual_binary"] == "Industrial") &
    (behavior_eval["behavior_label"] == "Transient")
].copy()

print(
    "Industrial events classified Transient:",
    len(industrial_transient)
)

display(
    industrial_transient[
        [
            "event_id",
            "actual_class_clean",
            "active_days",
            "duration_days",
            "detection_count",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km",
            "behavior_label"
        ]
    ].sort_values("max_frp", ascending=False)
)

Industrial events classified Transient: 14


,event_id,actual_class_clean,active_days,duration_days,detection_count,mean_frp,max_frp,spatial_diameter_km,behavior_label
57,3512,Industrial,1,1,1,3.160000,3.16,0.000000,Transient
50,274,Industrial,6,6,7,1.907143,3.00,0.558397,Transient
26,3482,Industrial,3,5,5,1.032000,1.70,0.441846,Transient
40,3976,Industrial,3,5,3,1.230000,1.47,0.141056,Transient
25,4077,Industrial,4,6,6,1.051667,1.19,0.449947,Transient
11,1181,Industrial,1,1,1,1.040000,1.04,0.000000,Transient
3,479,Industrial,2,2,2,0.790000,0.96,0.259333,Transient
33,3340,Industrial,1,1,1,0.870000,0.87,0.000000,Transient
55,4642,Industrial,1,1,1,0.850000,0.85,0.000000,Transient
58,3943,Industrial,2,2,2,0.655000,0.83,0.203472,Transient


In [113]:
# ============================================================
# CELL 96 — LOCATE WORLDCOVER DATA
# ============================================================

for name in list(globals()):
    obj = globals()[name]

    if isinstance(obj, pd.DataFrame):
        wc_cols = [
            str(c) for c in obj.columns
            if isinstance(c, str) and c.startswith("wc_")
        ]

        if wc_cols:
            print(
                f"{name}: shape={obj.shape}, "
                f"WorldCover columns={len(wc_cols)}"
            )
            print(wc_cols)

validation_geo: shape=(60, 82), WorldCover columns=11
['wc_tree_cover', 'wc_shrubland', 'wc_grassland', 'wc_cropland', 'wc_built_up', 'wc_bare_sparse', 'wc_snow_ice', 'wc_permanent_water', 'wc_wetland', 'wc_mangroves', 'wc_moss_lichen']
binary_df: shape=(15, 87), WorldCover columns=11
['wc_tree_cover', 'wc_shrubland', 'wc_grassland', 'wc_cropland', 'wc_built_up', 'wc_bare_sparse', 'wc_snow_ice', 'wc_permanent_water', 'wc_wetland', 'wc_mangroves', 'wc_moss_lichen']
evidence_table: shape=(12, 28), WorldCover columns=7
['wc_tree_cover', 'wc_shrubland', 'wc_grassland', 'wc_cropland', 'wc_built_up', 'wc_bare_sparse', 'wc_permanent_water']
validation_review: shape=(60, 20), WorldCover columns=5
['wc_built_up', 'wc_cropland', 'wc_tree_cover', 'wc_grassland', 'wc_bare_sparse']
industrial_events: shape=(11, 82), WorldCover columns=11
['wc_tree_cover', 'wc_shrubland', 'wc_grassland', 'wc_cropland', 'wc_built_up', 'wc_bare_sparse', 'wc_snow_ice', 'wc_permanent_water', 'wc_wetland', 'wc_mangroves'

In [114]:
# ============================================================
# CELL 97 — CORRECT 60-EVENT MODEL TABLE
# ============================================================

# Take the NEW 60-event validation_geo as the base.
# It already contains OSM + WorldCover context.

validation_full = validation_geo.copy()

# Attach behavior prediction from the complete 4,893-event dataset
behavior_map = full_events[
    [
        "event_id",
        "behavior_cluster",
        "behavior_label"
    ]
].drop_duplicates("event_id")

validation_full = validation_full.drop(
    columns=["behavior_cluster", "behavior_label"],
    errors="ignore"
).merge(
    behavior_map,
    on="event_id",
    how="left"
)

# Clean labels
validation_full["actual_class_clean"] = (
    validation_full["actual_class"]
    .astype("string")
    .str.strip()
)

validation_full["actual_binary"] = (
    validation_full["actual_class_clean"]
    .map({
        "Industrial": "Industrial",
        "Agricultural": "Non_Industrial",
        "Natural_Vegetation": "Non_Industrial",
        "Other": "Non_Industrial"
    })
)

print("Validation shape:", validation_full.shape)

print(
    "Missing behavior labels:",
    validation_full["behavior_label"].isna().sum()
)

print("\nClass distribution:")
display(
    validation_full["actual_class_clean"].value_counts()
)

print("\nBinary distribution:")
display(
    validation_full["actual_binary"].value_counts(dropna=False)
)

Validation shape: (60, 83)
Missing behavior labels: 0

Class distribution:


actual_class_clean
Industrial            11
Natural_Vegetation     4
Unknown                3
Name: count, dtype: Int64


Binary distribution:


actual_binary
NaN               45
Industrial        11
Non_Industrial     4
Name: count, dtype: int64

In [115]:
# ============================================================
# CELL 98 — FINAL CONTEXT CHECK
# ============================================================

required_context = [
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_works_km",
    "nearest_mine_km",
    "nearest_brick_km",
    "wc_built_up",
    "wc_tree_cover",
    "wc_grassland",
    "wc_cropland"
]

check = pd.DataFrame({
    "feature": required_context,
    "exists": [
        c in validation_full.columns
        for c in required_context
    ],
    "missing_values": [
        validation_full[c].isna().sum()
        if c in validation_full.columns
        else None
        for c in required_context
    ]
})

display(check)

assert len(validation_full) == 60
assert validation_full["behavior_label"].notna().all()
assert check["exists"].all()

print("60-EVENT VALIDATION TABLE READY")

,feature,exists,missing_values
0,nearest_industrial_zone_km,True,0
1,nearest_factory_km,True,0
2,nearest_works_km,True,0
3,nearest_mine_km,True,0
4,nearest_brick_km,True,0
5,wc_built_up,True,0
6,wc_tree_cover,True,0
7,wc_grassland,True,0
8,wc_cropland,True,0


60-EVENT VALIDATION TABLE READY


In [117]:
# ============================================================
# CELL 99 — RELOAD FULLY LABELED VALIDATION WORKBOOK
# ============================================================

import pandas as pd

validation_file = "VIIRS_60_Event_Validation_Review_Pack (1).xlsx"

validation_review_new = pd.read_excel(
    validation_file,
    sheet_name="Validation Review"
)

print("Shape:", validation_review_new.shape)
print("\nActual classes:")
display(
    validation_review_new["actual_class"].value_counts(dropna=False)
)

print("\nMissing labels:",
      validation_review_new["actual_class"].isna().sum())

print("Unique event IDs:",
      validation_review_new["event_id"].nunique())

Shape: (60, 31)

Actual classes:


actual_class
Industrial            32
Natural_Vegetation    15
Unknown                5
Industrial             3
Agricultural           3
Other                  2
Name: count, dtype: int64


Missing labels: 0
Unique event IDs: 60


In [119]:
# CELL 100 — NORMALIZE VALIDATION LABELS

validation_review_new["actual_class"] = (
    validation_review_new["actual_class"]
    .astype("string")
    .str.strip()
)

print("Normalized classes:")
display(
    validation_review_new["actual_class"].value_counts(dropna=False)
)

Normalized classes:


actual_class
Industrial            35
Natural_Vegetation    15
Unknown                5
Agricultural           3
Other                  2
Name: count, dtype: Int64

In [120]:
# CELL 101 — CREATE FINAL BINARY LABEL

validation_review_new["actual_binary"] = (
    validation_review_new["actual_class"]
    .map({
        "Industrial": "Industrial",
        "Agricultural": "Non_Industrial",
        "Natural_Vegetation": "Non_Industrial",
        "Other": "Non_Industrial"
    })
)

print("Binary validation:")
display(
    validation_review_new["actual_binary"].value_counts(dropna=False)
)

Binary validation:


actual_binary
Industrial        35
Non_Industrial    20
NaN                5
Name: count, dtype: int64

In [121]:
# ============================================================
# CELL 102 — FINAL VALIDATION DATASET
# ============================================================

# Start from the authoritative 60-event validation labels
validation_final = validation_review_new.copy()

# Add behavior predictions from complete V11 dataset
behavior_cols = [
    "event_id",
    "behavior_cluster",
    "behavior_label"
]

validation_final = validation_final.merge(
    full_events[behavior_cols],
    on="event_id",
    how="left"
)

# Add OSM + WorldCover from the existing 60-event contextual table
context_cols = [
    "event_id"
] + [
    c for c in validation_geo.columns
    if c.startswith("nearest_") or c.startswith("wc_")
]

validation_final = validation_final.merge(
    validation_geo[context_cols],
    on="event_id",
    how="left"
)

print("Final validation shape:", validation_final.shape)

print("\nActual classes:")
display(validation_final["actual_class"].value_counts())

print("\nBinary classes:")
display(validation_final["actual_binary"].value_counts(dropna=False))

print("\nMissing behavior:", validation_final["behavior_label"].isna().sum())

print("\nMissing contextual rows:",
      validation_final[
          [c for c in context_cols if c != "event_id"]
      ].isna().all(axis=1).sum())

Final validation shape: (60, 53)

Actual classes:


actual_class
Industrial            35
Natural_Vegetation    15
Unknown                5
Agricultural           3
Other                  2
Name: count, dtype: Int64


Binary classes:


actual_binary
Industrial        35
Non_Industrial    20
NaN                5
Name: count, dtype: int64


Missing behavior: 0

Missing contextual rows: 0


In [122]:
# ============================================================
# CELL 103 — BEHAVIOR-ONLY VALIDATION
# ============================================================

eval_df = validation_final[
    validation_final["actual_binary"].notna()
].copy()

# K-Means semantic label
# behavior_label is already Transient / Persistent

print("Evaluated events:", len(eval_df))

print("\nActual class × behavior:")
display(
    pd.crosstab(
        eval_df["actual_binary"],
        eval_df["behavior_label"],
        margins=True
    )
)

print("\nBehavior distribution:")
display(
    eval_df["behavior_label"].value_counts()
)

Evaluated events: 55

Actual class × behavior:


behavior_label,Persistent,Transient,All
actual_binary,,,
Industrial,21,14,35
Non_Industrial,1,19,20
All,22,33,55



Behavior distribution:


behavior_label
Transient     33
Persistent    22
Name: count, dtype: int64

In [123]:
# ============================================================
# CELL 104 — BEHAVIORAL METRICS
# ============================================================

from sklearn.metrics import accuracy_score, confusion_matrix

# Persistent = model predicts Industrial
eval_df["behavior_prediction"] = np.where(
    eval_df["behavior_label"] == "Persistent",
    "Industrial",
    "Non_Industrial"
)

accuracy = accuracy_score(
    eval_df["actual_binary"],
    eval_df["behavior_prediction"]
)

cm = confusion_matrix(
    eval_df["actual_binary"],
    eval_df["behavior_prediction"],
    labels=["Industrial", "Non_Industrial"]
)

print(f"Behavior-only accuracy: {accuracy:.3f}")

print("\nConfusion matrix:")
display(
    pd.DataFrame(
        cm,
        index=["Actual Industrial", "Actual Non-Industrial"],
        columns=["Pred Industrial", "Pred Non-Industrial"]
    )
)

Behavior-only accuracy: 0.727

Confusion matrix:


,Pred Industrial,Pred Non-Industrial
Actual Industrial,21,14
Actual Non-Industrial,1,19


In [124]:
# ============================================================
# CELL 105 — BEHAVIOR ERROR ANALYSIS
# ============================================================

errors = eval_df[
    eval_df["actual_binary"] != eval_df["behavior_prediction"]
].copy()

print("Total errors:", len(errors))

print("\nError types:")
display(
    pd.crosstab(
        errors["actual_binary"],
        errors["behavior_label"]
    )
)

print("\nIndustrial events missed by behavior:")
industrial_missed = eval_df[
    (eval_df["actual_binary"] == "Industrial") &
    (eval_df["behavior_label"] == "Transient")
].copy()

print("Count:", len(industrial_missed))

display(
    industrial_missed[
        [
            "event_id",
            "behavior_label",
            "active_days",
            "duration_days",
            "detection_count",
            "mean_frp",
            "max_frp",
            "spatial_diameter_km"
        ]
    ].sort_values("active_days", ascending=False)
)

Total errors: 15

Error types:


behavior_label,Persistent,Transient
actual_binary,,
Industrial,0,14
Non_Industrial,1,0



Industrial events missed by behavior:
Count: 14


,event_id,behavior_label,active_days,duration_days,detection_count,mean_frp,max_frp,spatial_diameter_km
50,274,Transient,6,6,7,1.907143,3.00,0.558397
25,4077,Transient,4,6,6,1.051667,1.19,0.449947
40,3976,Transient,3,5,3,1.230000,1.47,0.141056
42,1655,Transient,3,5,3,0.593333,0.66,0.437301
26,3482,Transient,3,5,5,1.032000,1.70,0.441846
3,479,Transient,2,2,2,0.790000,0.96,0.259333
58,3943,Transient,2,2,2,0.655000,0.83,0.203472
14,3906,Transient,1,1,1,0.670000,0.67,0.000000
11,1181,Transient,1,1,1,1.040000,1.04,0.000000
9,693,Transient,1,1,1,0.660000,0.66,0.000000


In [125]:
# ============================================================
# CELL 106 — CONTEXT: INDUSTRIAL vs NON-INDUSTRIAL
# ============================================================

context_features = [
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_works_km",
    "nearest_mine_km",
    "nearest_brick_km",
    "nearest_depot_km",
    "nearest_power_km",
    "nearest_other_industry_km",
    "wc_built_up",
    "wc_tree_cover",
    "wc_grassland",
    "wc_cropland",
    "wc_bare_sparse"
]

context_eval = eval_df[
    eval_df["behavior_label"] == "Transient"
].copy()

print("Transient events:", len(context_eval))

display(
    context_eval.groupby("actual_binary")[context_features]
    .median()
    .T
)

Transient events: 33


actual_binary,Industrial,Non_Industrial
nearest_industrial_zone_km,130.060121,205.112865
nearest_factory_km,250.610630,281.249070
nearest_works_km,56.298848,60.223701
nearest_mine_km,963.102650,1416.422927
nearest_brick_km,503.324800,293.925799
nearest_depot_km,365.383652,291.382983
nearest_power_km,NaN,NaN
nearest_other_industry_km,244.247422,305.076889
wc_built_up,0.000000,0.000000
wc_tree_cover,0.000000,0.000000


In [126]:
# ============================================================
# CELL 107 — MISSED INDUSTRIAL CONTEXT
# ============================================================

display(
    industrial_missed[
        ["event_id"] + context_features
    ].sort_values(
        "nearest_works_km",
        na_position="last"
    )
)

,event_id,nearest_industrial_zone_km,nearest_factory_km,nearest_works_km,nearest_mine_km,nearest_brick_km,nearest_depot_km,nearest_power_km,nearest_other_industry_km,wc_built_up,wc_tree_cover,wc_grassland,wc_cropland,wc_bare_sparse
50,274,9.683577,141.162420,4.169125,1119.766256,179.676357,46.882536,NaN,9.683577,1,0,0,0,0
57,3512,11.900746,620.693907,4.458287,1892.708746,606.442779,106.827964,NaN,11.900746,1,0,0,0,0
3,479,132.188123,272.836367,16.428585,1270.549571,301.233213,132.188123,NaN,258.272316,0,0,0,0,1
55,4642,41.036152,109.950277,17.109150,1149.547675,149.334794,74.622693,NaN,41.036152,1,0,0,0,0
9,693,50.006706,404.604189,24.223901,2017.129658,529.438181,260.604152,NaN,50.006706,0,1,0,0,0
33,3340,81.746053,81.746053,29.877789,788.227479,567.897526,432.819309,NaN,358.313916,1,0,0,0,0
42,1655,127.932119,286.091109,51.816845,742.022452,793.228208,679.013660,NaN,127.932119,0,0,0,0,1
25,4077,146.276932,146.353490,60.780850,721.591090,554.363459,360.720266,NaN,356.615752,0,0,0,0,1
26,3482,154.777922,154.868166,61.188408,709.992332,565.684299,370.047038,NaN,365.893665,0,0,1,0,0
11,1181,191.334864,340.754117,66.524461,1787.911040,309.779307,229.467075,NaN,230.222527,1,0,0,0,0


In [127]:
# ============================================================
# CELL 108 — OSM RESCUE ANALYSIS
# ============================================================

osm_features = [
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_works_km",
    "nearest_mine_km",
    "nearest_brick_km",
    "nearest_depot_km",
    "nearest_other_industry_km"
]

rescue_df = eval_df[
    eval_df["behavior_label"] == "Transient"
].copy()

for radius in [1, 3, 5, 10, 25]:
    rescue_df[f"osm_within_{radius}km"] = (
        rescue_df[osm_features]
        .le(radius)
        .any(axis=1)
    )

print("Transient events with OSM evidence:")
for radius in [1, 3, 5, 10, 25]:
    col = f"osm_within_{radius}km"

    print(f"\n{radius} km:")
    display(
        pd.crosstab(
            rescue_df["actual_binary"],
            rescue_df[col],
            margins=True
        )
    )

Transient events with OSM evidence:

1 km:


osm_within_1km,False,All
actual_binary,,
Industrial,14,14
Non_Industrial,19,19
All,33,33



3 km:


osm_within_3km,False,All
actual_binary,,
Industrial,14,14
Non_Industrial,19,19
All,33,33



5 km:


osm_within_5km,False,True,All
actual_binary,,,
Industrial,12,2,14
Non_Industrial,19,0,19
All,31,2,33



10 km:


osm_within_10km,False,True,All
actual_binary,,,
Industrial,12,2,14
Non_Industrial,18,1,19
All,30,3,33



25 km:


osm_within_25km,False,True,All
actual_binary,,,
Industrial,9,5,14
Non_Industrial,17,2,19
All,26,7,33


In [128]:
# ============================================================
# CELL 108 — INSPECT OSM-POSITIVE TRANSIENT EVENTS
# ============================================================

for radius in [5, 10, 25]:
    col = f"osm_within_{radius}km"

    print(f"\n========== {radius} KM ==========")

    cols = [
        "event_id",
        "actual_binary",
        "behavior_label",
        "active_days",
        "mean_frp",
        "max_frp"
    ] + osm_features

    display(
        rescue_df.loc[
            rescue_df[col],
            cols
        ].sort_values("actual_binary")
    )


========== 5 KM ==========


,event_id,actual_binary,behavior_label,active_days,mean_frp,max_frp,nearest_industrial_zone_km,nearest_factory_km,nearest_works_km,nearest_mine_km,nearest_brick_km,nearest_depot_km,nearest_other_industry_km
50,274,Industrial,Transient,6,1.907143,3.00,9.683577,141.162420,4.169125,1119.766256,179.676357,46.882536,9.683577
57,3512,Industrial,Transient,1,3.160000,3.16,11.900746,620.693907,4.458287,1892.708746,606.442779,106.827964,11.900746



========== 10 KM ==========


,event_id,actual_binary,behavior_label,active_days,mean_frp,max_frp,nearest_industrial_zone_km,nearest_factory_km,nearest_works_km,nearest_mine_km,nearest_brick_km,nearest_depot_km,nearest_other_industry_km
50,274,Industrial,Transient,6,1.907143,3.00,9.683577,141.162420,4.169125,1119.766256,179.676357,46.882536,9.683577
57,3512,Industrial,Transient,1,3.160000,3.16,11.900746,620.693907,4.458287,1892.708746,606.442779,106.827964,11.900746
39,1686,Non_Industrial,Transient,1,6.153333,11.61,15.288319,281.249070,7.020940,1720.503939,428.729743,291.382983,139.782410



========== 25 KM ==========


,event_id,actual_binary,behavior_label,active_days,mean_frp,max_frp,nearest_industrial_zone_km,nearest_factory_km,nearest_works_km,nearest_mine_km,nearest_brick_km,nearest_depot_km,nearest_other_industry_km
3,479,Industrial,Transient,2,0.790000,0.96,132.188123,272.836367,16.428585,1270.549571,301.233213,132.188123,258.272316
9,693,Industrial,Transient,1,0.660000,0.66,50.006706,404.604189,24.223901,2017.129658,529.438181,260.604152,50.006706
50,274,Industrial,Transient,6,1.907143,3.00,9.683577,141.162420,4.169125,1119.766256,179.676357,46.882536,9.683577
55,4642,Industrial,Transient,1,0.850000,0.85,41.036152,109.950277,17.109150,1149.547675,149.334794,74.622693,41.036152
57,3512,Industrial,Transient,1,3.160000,3.16,11.900746,620.693907,4.458287,1892.708746,606.442779,106.827964,11.900746
38,2277,Non_Industrial,Transient,1,0.430000,0.43,22.779210,186.375501,47.332893,1965.879972,512.085578,109.930753,116.833171
39,1686,Non_Industrial,Transient,1,6.153333,11.61,15.288319,281.249070,7.020940,1720.503939,428.729743,291.382983,139.782410


In [129]:
# ============================================================
# CELL 109 — OSM EVIDENCE ACROSS BEHAVIOR CLASSES
# ============================================================

osm_test = eval_df.copy()

for radius in [1, 3, 5, 10]:
    osm_test[f"osm_{radius}km"] = (
        osm_test[osm_features].le(radius).any(axis=1)
    )

summary = []

for behavior in ["Persistent", "Transient"]:
    for actual in ["Industrial", "Non_Industrial"]:

        subset = osm_test[
            (osm_test["behavior_label"] == behavior) &
            (osm_test["actual_binary"] == actual)
        ]

        row = {
            "behavior": behavior,
            "actual": actual,
            "events": len(subset)
        }

        for radius in [1, 3, 5, 10]:
            row[f"osm_{radius}km_count"] = (
                subset[f"osm_{radius}km"].sum()
            )

        summary.append(row)

display(pd.DataFrame(summary))

,behavior,actual,events,osm_1km_count,osm_3km_count,osm_5km_count,osm_10km_count
0,Persistent,Industrial,21,0,0,0,2
1,Persistent,Non_Industrial,1,0,1,1,1
2,Transient,Industrial,14,0,0,2,2
3,Transient,Non_Industrial,19,0,0,0,1


In [137]:
# ============================================================
# CELL 110 — FINAL DOMAIN CLASSIFIER
# ============================================================

final_eval = eval_df.copy()

# K-Means cluster semantics from the frozen behavior model
# Cluster 0 = Transient
# Cluster 1 = Persistent

final_eval["behavior_type"] = final_eval["behavior_cluster"].map({
    0: "Transient",
    1: "Persistent"
})

osm_features = [
    "nearest_industrial_zone_km",
    "nearest_factory_km",
    "nearest_works_km",
    "nearest_mine_km",
    "nearest_brick_km",
    "nearest_depot_km",
    "nearest_power_km",
    "nearest_other_industry_km"
]

# Positive OSM evidence:
# any known industrial entity within 5 km
final_eval["osm_industrial_evidence"] = (
    final_eval[osm_features].le(5).any(axis=1)
)

def classify_event(row):

    # Persistent thermal behaviour
    if row["behavior_type"] == "Persistent":
        return "Industrial"

    # Transient + nearby industrial infrastructure
    if (
        row["behavior_type"] == "Transient"
        and row["osm_industrial_evidence"]
    ):
        return "Industrial"

    # Insufficient evidence
    return "Uncertain"


final_eval["final_prediction"] = final_eval.apply(
    classify_event,
    axis=1
)

print("Behavior types:")
print(final_eval["behavior_type"].value_counts())

print("\nFinal predictions:")
print(final_eval["final_prediction"].value_counts())

Behavior types:
behavior_type
Transient     33
Persistent    22
Name: count, dtype: int64

Final predictions:
final_prediction
Uncertain     31
Industrial    24
Name: count, dtype: int64


In [138]:
# ============================================================
# CELL 111 — FINAL VALIDATION
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

total_labeled = len(final_eval)

scored = final_eval[
    final_eval["final_prediction"].isin(
        ["Industrial", "Non_Industrial"]
    )
].copy()

industrial_scored = (
    scored["final_prediction"] == "Industrial"
).sum()

industrial_correct = (
    (scored["actual_binary"] == "Industrial") &
    (scored["final_prediction"] == "Industrial")
).sum()

uncertain = (
    final_eval["final_prediction"] == "Uncertain"
).sum()

print("Total labeled events:", total_labeled)
print("Confidently classified:", len(scored))
print("Uncertain:", uncertain)
print("Coverage:", round(len(scored) / total_labeled * 100, 2), "%")

print("\nConfident prediction performance:")
print(
    classification_report(
        scored["actual_binary"],
        scored["final_prediction"],
        labels=["Industrial", "Non_Industrial"],
        zero_division=0
    )
)

print("\nConfusion matrix:")
display(
    pd.crosstab(
        scored["actual_binary"],
        scored["final_prediction"],
        rownames=["Actual"],
        colnames=["Predicted"],
        margins=True
    )
)

print("\nIndustrial precision among confident predictions:",
      round(industrial_correct / industrial_scored * 100, 2), "%")

Total labeled events: 55
Confidently classified: 24
Uncertain: 31
Coverage: 43.64 %

Confident prediction performance:
                precision    recall  f1-score   support

    Industrial       0.96      1.00      0.98        23
Non_Industrial       0.00      0.00      0.00         1

      accuracy                           0.96        24
     macro avg       0.48      0.50      0.49        24
  weighted avg       0.92      0.96      0.94        24


Confusion matrix:


Predicted,Industrial,All
Actual,,
Industrial,23,23
Non_Industrial,1,1
All,24,24



Industrial precision among confident predictions: 95.83 %
